In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Install R in Python runtime
!apt-get install -y libcurl4-openssl-dev libssl-dev libxml2-dev
%load_ext rpy2.ipython


In [ ]:
!apt-get update
!apt-get install -y build-essential libcurl4-openssl-dev libssl-dev libxml2-dev


In [ ]:
!apt-get install -y libfontconfig1-dev libcairo2-dev

In [ ]:
%%R
install.packages("ggridges", dependencies = FALSE)

In [ ]:
%%R
install.packages("Seurat")

In [ ]:
%%R
# Install CRAN packages
if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}
if (!requireNamespace("devtools", quietly = TRUE)) {
  install.packages("devtools")
}
if (!requireNamespace("Seurat", quietly = TRUE)) {
  install.packages("Seurat")
}
if (!requireNamespace("Matrix", quietly = TRUE)) {
  install.packages("Matrix")
}
if (!requireNamespace("dplyr", quietly = TRUE)) {
  install.packages("dplyr")
}
if (!requireNamespace("patchwork", quietly = TRUE)) {
  install.packages("patchwork")
}

# Install Bioconductor packages
if (!requireNamespace("SingleR", quietly = TRUE)) {
  BiocManager::install("SingleR")
}
if (!requireNamespace("celldex", quietly = TRUE)) {
  BiocManager::install("celldex")
}


In [ ]:
%%R
install.packages(c("Rcpp", "RcppArmadillo", "devtools", "remotes"))


In [ ]:
%%R
# Install BayesPrism dependencies
if (!requireNamespace("snowfall", quietly = TRUE)) {
  install.packages("snowfall")
}
if (!requireNamespace("NMF", quietly = TRUE)) {
  install.packages("NMF")
}
if (!requireNamespace("scran", quietly = TRUE)) {
  BiocManager::install("scran")
}
if (!requireNamespace("gplots", quietly = TRUE)) {
  install.packages("gplots")
}

In [ ]:
%%R
devtools::install_github("Danko-Lab/BayesPrism",
                         subdir = "BayesPrism",
                         dependencies = FALSE,
                         force = TRUE)


In [ ]:
%%R

# Load libraries
library(Seurat)
library(Matrix)
library(dplyr)
library(patchwork)
library(SingleR)
library(celldex)
library(BayesPrism)
library(ggplot2)

In [ ]:
%%R
# ---- BLOCK 0: CONFIG ----
USE_SAVED_COPYKAT <- TRUE
COPYKAT_DIR <- "/content/drive/MyDrive/BayesPrism/copykat_results"

# Dedicated folder for this ECM run
ECM_RUN_DIR <- "/content/drive/MyDrive/BayesPrism/two_phase_results_ecm"
dir.create(ECM_RUN_DIR, recursive = TRUE, showWarnings = FALSE)


In [ ]:
%%R
# STEP 1: LOAD AND PROCESS COMPLETE TUMOR scRNA-seq DATA
cat("=== LOADING COMPLETE TUMOR scRNA-seq DATA ===\n")

suppressPackageStartupMessages({
  library(Seurat); library(Matrix)
})

# --- paths ---
tumor_data_dir <- "/content/drive/MyDrive/BayesPrism/GSE242230"
results_dir    <- "/content/drive/MyDrive/BayesPrism/two_phase_results"
dir.create(results_dir, recursive = TRUE, showWarnings = FALSE)

# ReadMtx
load_10x_sample <- function(sample_id, base_dir){
  mtx  <- file.path(base_dir, paste0(sample_id, "_matrix.mtx.gz"))
  feat <- file.path(base_dir, paste0(sample_id, "_features.tsv.gz"))
  bc   <- file.path(base_dir, paste0(sample_id, "_barcodes.tsv.gz"))
  if (!file.exists(mtx) || !file.exists(feat) || !file.exists(bc)) {
    cat("Skipping", sample_id, "- missing one or more files\n"); return(NULL)
  }
  mat <- Seurat::ReadMtx(mtx = mtx, features = feat, cells = bc)
  so  <- CreateSeuratObject(counts = mat, project = sample_id, min.cells = 3, min.features = 200)
  so$sample_id  <- sample_id
  so$orig.ident <- sample_id
  cat("Loaded", sample_id, ":", ncol(so), "cells,", nrow(so), "genes\n")
  so
}

# Find sample IDs from *_matrix.mtx.gz
mtx_files  <- list.files(tumor_data_dir, pattern = "GSM.*_matrix\\.mtx\\.gz$", full.names = TRUE)
sample_ids <- sub("_matrix\\.mtx\\.gz$", "", basename(mtx_files))
cat("Found", length(sample_ids), "samples\n")

sample_objects <- list()
for (sid in sample_ids) {
  obj <- try(load_10x_sample(sid, tumor_data_dir), silent = TRUE)
  if (!inherits(obj, "try-error") && !is.null(obj)) sample_objects[[sid]] <- obj
  if (length(sample_objects) %% 5 == 0) cat("Loaded", length(sample_objects), "samples so far...\n")
}
cat("Successfully loaded", length(sample_objects), "samples\n")


In [ ]:
%%R
# STEP 2: MERGE AND QC
cat("=== MERGING AND QC ===\n")

stopifnot(length(sample_objects) >= 1)
if (length(sample_objects) == 1) {
  tumor_combined <- sample_objects[[1]]
} else {
  tumor_combined <- merge(sample_objects[[1]],
                          y = sample_objects[2:length(sample_objects)],
                          add.cell.ids = names(sample_objects),
                          project = "PDAC_Tumor_TwoPhase")
}
rm(sample_objects); gc()

cat("Combined object:", ncol(tumor_combined), "cells,", nrow(tumor_combined), "genes\n")

# QC metrics
rn <- rownames(tumor_combined)
mt_pat <- if (any(grepl("^MT-", rn))) "^MT-" else "^mt-"
tumor_combined[["percent.mt"]]   <- PercentageFeatureSet(tumor_combined, pattern = mt_pat)
tumor_combined[["percent.ribo"]] <- PercentageFeatureSet(tumor_combined, pattern = "^RP[SL]")

feats  <- tumor_combined$nFeature_RNA
counts <- tumor_combined$nCount_RNA
mt_pct <- tumor_combined$percent.mt

feat_lo <- max(200,  quantile(feats,  0.005, na.rm = TRUE))
feat_hi <- min(8000, quantile(feats,  0.995, na.rm = TRUE))
count_hi<-            quantile(counts, 0.99,  na.rm = TRUE)
mt_thr  <- min(25,    quantile(mt_pct, 0.95,  na.rm = TRUE))

cells_before <- ncol(tumor_combined)
tumor_combined <- subset(tumor_combined, subset =
  nFeature_RNA > feat_lo &
  nFeature_RNA < feat_hi &
  nCount_RNA   < count_hi &
  percent.mt   < mt_thr
)
cells_after <- ncol(tumor_combined)
cat("QC filtering: retained", cells_after, "of", cells_before, "cells\n")

# Normalize/scale on variable features only
tumor_combined <- NormalizeData(tumor_combined)
tumor_combined <- FindVariableFeatures(tumor_combined, selection.method = "vst", nfeatures = 2000)
tumor_combined <- ScaleData(tumor_combined,
                            features = VariableFeatures(tumor_combined),
                            vars.to.regress = c("percent.mt", "nCount_RNA"))

tumor_combined <- RunPCA(tumor_combined, npcs = 50, verbose = FALSE)
set.seed(42)
tumor_combined <- FindNeighbors(tumor_combined, dims = 1:30, verbose = FALSE)
tumor_combined <- FindClusters(tumor_combined, resolution = 0.5, verbose = FALSE)
tumor_combined <- RunUMAP(tumor_combined, dims = 1:30, verbose = FALSE)

# --- CRITICAL: guarantee sample_id exists and is a named vector (for split()) ---
if (is.null(tumor_combined$sample_id)) {
  if ("orig.ident" %in% colnames(tumor_combined@meta.data)) {
    tumor_combined$sample_id <- tumor_combined$orig.ident
  } else {
    tumor_combined$sample_id <- "sample1"
  }
}
tumor_combined$sample_id <- as.character(tumor_combined$sample_id)
sid <- tumor_combined$sample_id
names(sid) <- colnames(tumor_combined)
sid[is.na(sid)] <- "sample_unknown"
stopifnot(length(sid) == ncol(tumor_combined), identical(names(sid), colnames(tumor_combined)))

cat("Sample IDs present across", length(unique(sid)), "samples\n")


In [ ]:
%%R
# Seurat v5: join layers after merge/QC
tumor_combined <- JoinLayers(tumor_combined, layers = NULL)


In [ ]:
%%R

cat("=== ENHANCED NON-OVERLAPPING MARKER DEFINITIONS (Tumor + ECM/Stroma) ===\n")

# Tumor / epithelial marker panels
tumor_markers <- list(
  epithelial_general   = c("EPCAM","KRT8","KRT18","KRT19","KRT7","CDH1"),
  pancreatic_ductal    = c("CFTR","CA2","MUC1","ANXA4","SOX9","SLC4A4","BICC1","GLIS3","SCTR","TFF3"),
  tumor_classical      = c("TFF1","LGALS4","TSPAN8","REG4","ST6GALNAC1","CTSE","GATA6","HNF1A","HNF4A","FOXA2","FOXA3","PDX1"),
  tumor_basal_like     = c("S100A2","KRT6A","KRT17","SERPINB4","LY6D","HMGA2","TP63","KRT5","DSC3","PKP1"),
  proliferation        = c("MKI67","TOP2A","PCNA","CDK1","CCNB1","FOXM1","E2F1","E2F2","E2F3"),
  stemness             = c("CD44","PROM1","ALDH1A1","BMI1","NANOG"),
  invasion_emt         = c("VIM","SNAI1","SNAI2","ZEB1","ZEB2","CDH2","S100A4"),
  normal_ductal        = c("CFTR","BICC1","SLC4A4","GLIS3","SCTR","AMBP","CA2"),
  normal_acinar        = c("PRSS1","CPA1","CTRB1","CELA3A","AMY2A","PNLIP","CTRB2","CTRC"),
  normal_islet         = c("INS","GCG","SST","PPY","IAPP","PDX1","CHGA","SYP")
)

# Stromal / ECM marker panels
stromal_markers <- list(
  fibroblasts_general    = c("COL1A1","COL1A2","COL3A1","DCN","LUM","BGN"),
  myCAF                  = c("ACTA2","TAGLN","MYL9","POSTN","MMP11","HOPX","MYH11","CNN1","MYLK"),
  iCAF                   = c("IL6","CXCL12","CXCL14","CCL2","PDGFRA","CFD","PLA2G2A","CD34"),
  apCAF                  = c("HLA-DRA","HLA-DPB1","CD74","CIITA","SLPI","HLA-DPA1"),
  caf_classical          = c("PDGFRA","PDGFRB","THY1","FAP","PDPN","CAV1","LRRC15","CDH11"),
  caf_activated          = c("ACTA2","TAGLN","CNN1","MYH11","MYLK","POSTN"),
  stellate_cells         = c("RGS5","PDGFRB","CSPG4","MCAM","NOTCH3","ABCC9","GFAP","SPARC"),
  pancreatic_stellate    = c("GFAP","VIM","PDGFRB","ACTA2","TIMP1","COL1A1","SPARC","DES"),
  ecm_collagen           = c("COL1A1","COL1A2","COL3A1","COL4A1","COL5A1","COL6A1","COL18A1"),
  ecm_fibronectin        = c("FN1","FNDC1","EFEMP1","THBS1","THBS2"),
  ecm_proteoglycans      = c("DCN","LUM","BGN","FMOD","PRELP","OGN"),
  ecm_remodeling         = c("MMP2","MMP9","MMP11","MMP14","TIMP1","TIMP2")
)

# Vascular / mural marker panels
vascular_markers <- list(
  endothelial_general    = c("PECAM1","VWF","CDH5","KDR","FLT1","FLI1","ERG"),
  endothelial_arterial   = c("GJA5","BMX","DKK2","HEY1","EFNB2","RERGL"),
  endothelial_venous     = c("NR2F2","APLNR","EPHB4","NRP2","HMCN2","FLNC"),
  endothelial_capillary  = c("CA4","RGCC","PRSS23","FCN3","PLVAP","ESM1"),
  endothelial_lymphatic  = c("PROX1","LYVE1","FLT4","PDPN","CCL21"),
  endothelial_islet      = c("COL13A1","ESM1","PLVAP","UNC5B","LAMA4","KDR","THBS1","ACE","PASK","F2RL3","CXCR4"),
  endothelial_acinar     = c("GPIHBP1","AQP1","KLF4","KLF2","KDR"),
  pericytes              = c("RGS5","PDGFRB","CSPG4","MCAM","NOTCH3","ABCC9"),
  smooth_muscle          = c("ACTA2","MYH11","TAGLN","CNN1","MYLK","CASQ2","KCNAB1")
)

# Other non-immune cell panels (for QC)
other_markers <- list(
  neural_cells  = c("TUBB3","MAP2","SYP","CHGA","NCAM1","UCHL1"),
  schwann_cells = c("MPZ","PLP1","S100B","SOX10","NGFR","GFAP"),
  adipocytes    = c("ADIPOQ","LEP","PLIN1","FABP4","LPL")
)

# Negative gates (exclude immune/heme/mast)
negative_markers <- list(
  immune_cd45   = c("PTPRC","CD3D","CD3E","CD3G","TRAC","MS4A1","CD79A","NKG7","GNLY","PRF1",
                    "LYZ","S100A8","S100A9","LST1","FCGR3B","CD14","HLA-DRA","HLA-DPA1","HLA-DPB1"),
  rbc_platelet  = c("HBA1","HBA2","HBB","PPBP","PF4"),
  mast          = c("TPSAB1","CPA3","KIT")
)

# Assemble comprehensive_markers expected by downstream code
epithelial_ductal <- unique(setdiff(
  c(tumor_markers$epithelial_general, tumor_markers$pancreatic_ductal),
  "SPP1"
))
fibro_quiescent <- unique(c("DPT","DCN","LUM","COL1A1","COL1A2","COL3A1","BGN"))

comprehensive_markers <- list(
  epithelial_ductal      = epithelial_ductal,
  fibroblasts_quiescent  = fibro_quiescent,
  caf_myofibroblastic    = stromal_markers$myCAF,
  caf_inflammatory       = stromal_markers$iCAF,
  caf_antigen_presenting = stromal_markers$apCAF,
  endothelial_blood      = vascular_markers$endothelial_general,
  endothelial_lymphatic  = vascular_markers$endothelial_lymphatic,
  pericytes              = unique(c(vascular_markers$pericytes, vascular_markers$smooth_muscle)),
  acinar_cells           = tumor_markers$normal_acinar,
  mast_cells             = negative_markers$mast,
  epithelial_general     = epithelial_ductal,                      # alias
  basal_like             = tumor_markers$tumor_basal_like,         # alias
  classical              = tumor_markers$tumor_classical,          # alias
  stellate_cells         = unique(c(stromal_markers$stellate_cells,
                                    stromal_markers$pancreatic_stellate))
)

# Phase 2 immune-enhanced panel (empty; CD45+ excluded in this run)
immune_enhanced_markers <- list()

# QC-only marker sets
qc_markers <- list(
  normal_epithelial = tumor_markers$normal_ductal,
  proliferation     = c("MKI67","TOP2A","PCNA","CDK1"),
  platelets         = c("ITGA2B","PPBP","PF4"),
  erythrocytes      = c("HBA1","HBA2","HBB")
)

# Helper functions
calculate_marker_score <- function(markers, expr_matrix, method = "mean") {
  genes <- intersect(markers, rownames(expr_matrix))
  if (length(genes) == 0) return(rep(0, ncol(expr_matrix)))
  if (length(genes) == 1) return(as.numeric(expr_matrix[genes, ]))
  if (method == "mean")   return(colMeans(expr_matrix[genes, , drop = FALSE]))
  if (method == "median") return(apply(expr_matrix[genes, , drop = FALSE], 2, median))
  if (method == "pc1")    return(prcomp(t(expr_matrix[genes, , drop = FALSE]), scale. = TRUE)$x[,1])
  stop("Unknown method")
}

check_marker_overlaps <- function(marker_list, min_overlap = 2) {
  if (length(marker_list) < 2) return(list())
  overlaps <- list(); nms <- names(marker_list)
  for (i in seq_len(length(nms) - 1)) for (j in (i + 1):length(nms)) {
    ov <- intersect(marker_list[[nms[i]]], marker_list[[nms[j]]])
    if (length(ov) >= min_overlap) overlaps[[paste(nms[i], "vs", nms[j])]] <- ov
  }
  overlaps
}

# Overlap checks
cat("Checking Phase 1 overlaps...\n")
skip_alias <- c("epithelial_general")  # alias duplicates epithelial_ductal
phase1_names <- setdiff(names(comprehensive_markers), skip_alias)
phase1_overlaps <- check_marker_overlaps(comprehensive_markers[phase1_names], min_overlap = 2)
if (length(phase1_overlaps)) {
  cat("Phase 1 overlaps detected:\n")
  for (k in names(phase1_overlaps)) cat("  ", k, ":", paste(phase1_overlaps[[k]], collapse = ", "), "\n")
} else {
  cat("Phase 1: No significant overlaps detected\n")
}

cat("Checking Phase 2 overlaps...\n")
phase2_overlaps <- check_marker_overlaps(immune_enhanced_markers, min_overlap = 2)
if (length(phase2_overlaps)) {
  cat("Phase 2 overlaps detected:\n")
  for (k in names(phase2_overlaps)) cat("  ", k, ":", paste(phase2_overlaps[[k]], collapse = ", "), "\n")
} else {
  cat("Phase 2: No significant overlaps detected\n")
}

cat("Enhanced marker definitions created:\n")
cat("- Phase 1:", length(comprehensive_markers), "cell-type sets\n")
cat("- Phase 2:", length(immune_enhanced_markers), "immune subtype sets\n")
cat("- QC markers:", length(qc_markers), "validation sets\n")


In [ ]:
%%R
# STEP 3.5: FAST COPYKAT (EPITHELIAL-ONLY, PER-SAMPLE)
cat("=== Running COPYKAT ===\n")
#Install copykat

if (!requireNamespace("copykat", quietly = TRUE)) {
  devtools::install_github("navinlabcode/copykat")
}
library(copykat)

In [ ]:
%%R
suppressPackageStartupMessages({
  library(Seurat); library(Matrix); library(copykat)
})

cat("=== SEURAT V5 COPYKAT SOLUTION ===\n")
cat("Detected Seurat v5 with Assay5 - using layer-based approach\n")

# STEP 1: JOIN LAYERS TO CREATE UNIFIED MATRIX

cat("Joining count layers...\n")

tumor_combined <- JoinLayers(tumor_combined, layers = NULL)

cat("Accessing unified counts matrix...\n")
tryCatch({
  counts_matrix <- GetAssayData(tumor_combined, layer = "counts")
  cat("✓ Successfully accessed counts via layer method\n")
}, error = function(e) {

  tryCatch({
    counts_matrix <- tumor_combined@assays$RNA@layers$counts
    cat("✓ Successfully accessed counts via direct layer access\n")
  }, error = function(e2) {
    stop("Cannot access counts after joining layers")
  })
})

cat("Counts matrix:", nrow(counts_matrix), "genes x", ncol(counts_matrix), "cells\n")

# STEP 2: EPITHELIAL CELL IDENTIFICATION

cat("Identifying epithelial cells...\n")

# Define epithelial markers
epithelial_markers <- c("EPCAM", "KRT8", "KRT18", "KRT19", "KRT7")
available_epi <- intersect(epithelial_markers, rownames(counts_matrix))

if(length(available_epi) == 0) {
  stop("No epithelial markers found in the data")
}

cat("Using epithelial markers:", paste(available_epi, collapse = ", "), "\n")

# Calculate epithelial score using raw counts
if(length(available_epi) == 1) {
  epi_score <- as.numeric(counts_matrix[available_epi, ])
} else {
  epi_score <- Matrix::colSums(counts_matrix[available_epi, , drop = FALSE])
}
names(epi_score) <- colnames(counts_matrix)

# Per-sample epithelial gating
sample_ids <- tumor_combined$sample_id
names(sample_ids) <- colnames(tumor_combined)

by_sample_scores <- split(epi_score, sample_ids[names(epi_score)])
epithelial_cells <- unlist(lapply(names(by_sample_scores), function(samp) {
  scores <- by_sample_scores[[samp]]
  if(length(scores) < 50) return(names(scores))  # Keep all if too few cells

  # Keep top 70% of epithelial-expressing cells per sample
  threshold <- quantile(scores, 0.70)
  names(scores)[scores > threshold]
}))

cat("Epithelial cells identified:", length(epithelial_cells), "out of", ncol(counts_matrix), "\n")

# Downsample if too many cells
max_cells <- 8000
if(length(epithelial_cells) > max_cells) {
  set.seed(42)
  epithelial_cells <- sample(epithelial_cells, max_cells)
  cat("Downsampled to", length(epithelial_cells), "cells for CopyKAT\n")
}

# STEP 3: PREPARE MATRIX FOR COPYKAT

cat("Preparing matrix for CopyKAT...\n")

# Subset to epithelial cells
epi_counts <- counts_matrix[, epithelial_cells, drop = FALSE]

# Clean genes
gene_names <- rownames(epi_counts)

# Remove mitochondrial, ribosomal genes
mito_genes <- grep("^MT-|^mt-", gene_names, value = TRUE)
ribo_genes <- grep("^RP[SL]|^rp[sl]", gene_names, value = TRUE)
bad_genes <- unique(c(mito_genes, ribo_genes))

if(length(bad_genes) > 0) {
  epi_counts <- epi_counts[setdiff(gene_names, bad_genes), , drop = FALSE]
  cat("Removed", length(bad_genes), "mitochondrial/ribosomal genes\n")
}

# Remove duplicate gene names
dup_genes <- duplicated(rownames(epi_counts))
if(any(dup_genes)) {
  epi_counts <- epi_counts[!dup_genes, , drop = FALSE]
  cat("Removed", sum(dup_genes), "duplicate genes\n")
}

# Filter very lowly expressed genes
gene_counts <- Matrix::rowSums(epi_counts > 0)  # Number of cells expressing each gene
min_cells <- max(3, round(ncol(epi_counts) * 0.01))  # At least 1% of cells
keep_genes <- gene_counts >= min_cells
epi_counts <- epi_counts[keep_genes, , drop = FALSE]

cat("Final matrix for CopyKAT:", nrow(epi_counts), "genes x", ncol(epi_counts), "cells\n")

# STEP 4: RUN COPYKAT

cat("Running CopyKAT analysis...\n")

# Create output directory
out_dir <- "/content/copykat_outputs"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# Convert sparse to dense matrix
cat("Converting to dense matrix (this may take a moment)...\n")
epi_dense <- as.matrix(epi_counts)

# Clear memory
rm(epi_counts, counts_matrix); gc()

cat("Running CopyKAT with optimized parameters...\n")

# Run CopyKAT
ck_result <- tryCatch({
  copykat::copykat(
    rawmat = epi_dense,
    id.type = "S",                    # Symbols
    ngene.chr = 5,                    # Min genes per chromosome
    win.size = 25,                    # Window size for smoothing
    KS.cut = 0.1,                     # KS test cutoff
    sam.name = "PDAC_v5",             # Sample name
    distance = "euclidean",           # Distance metric
    norm.cell.names = "",             # Auto-detect normal cells
    n.cores = 2                       # Number of cores
  )
}, error = function(e) {
  cat("First attempt failed:", e$message, "\n")
  cat("Trying with more conservative parameters...\n")

  # conservative parameters
  copykat::copykat(
    rawmat = epi_dense,
    id.type = "S",
    ngene.chr = 3,                    # fewer genes per chr
    win.size = 50,                    # Larger window
    KS.cut = 0.15,                    # Higher cutoff
    sam.name = "PDAC_v5_conservative",
    n.cores = 1                       # Single core for stability
  )
})

if(is.null(ck_result)) {
  stop("CopyKAT failed with both parameter sets")
}

cat("✓ CopyKAT analysis completed successfully!\n")

# STEP 5: PROCESS AND ADD RESULTS

cat("Processing CopyKAT results...\n")

# Extract predictions
predictions <- ck_result$prediction
cat("Predictions generated for", nrow(predictions), "cells\n")
print(table(predictions$copykat.pred))

# Initialize all cells with default values
tumor_combined$copykat_pred <- "diploid"
tumor_combined$copykat_confidence <- 0.5
tumor_combined$malignant_call <- NA_character_

# Add predictions for analyzed cells
analyzed_cells <- intersect(rownames(predictions), colnames(tumor_combined))
tumor_combined$copykat_pred[analyzed_cells] <- predictions[analyzed_cells, "copykat.pred"]

# Add confidence scores if available
if("prob.aneuploid" %in% colnames(predictions)) {
  tumor_combined$copykat_confidence[analyzed_cells] <- predictions[analyzed_cells, "prob.aneuploid"]
}

# Create malignant classification for epithelial cells
epithelial_mask <- colnames(tumor_combined) %in% epithelial_cells
tumor_combined$malignant_call[epithelial_mask] <- ifelse(
  tumor_combined$copykat_pred[epithelial_mask] == "aneuploid",
  "Tumor_cells",
  "Normal_ductal"
)

# STEP 6: RESULTS SUMMARY AND SAVE

cat("\n=== COPYKAT RESULTS SUMMARY ===\n")
cat("Total cells in dataset:", ncol(tumor_combined), "\n")
cat("Epithelial cells analyzed:", length(epithelial_cells), "\n")
cat("Cells with CopyKAT predictions:", length(analyzed_cells), "\n")

# Final counts
malignant_count <- sum(tumor_combined$copykat_pred == "aneuploid", na.rm = TRUE)
normal_epi_count <- sum(tumor_combined$malignant_call == "Normal_ductal", na.rm = TRUE)
tumor_count <- sum(tumor_combined$malignant_call == "Tumor_cells", na.rm = TRUE)

cat("Malignant cells (aneuploid):", malignant_count, "\n")
cat("Normal epithelial cells:", normal_epi_count, "\n")
cat("Tumor cells:", tumor_count, "\n")

# Malignant fraction of epithelial cells
if(length(epithelial_cells) > 0) {
  malignant_fraction <- malignant_count / length(epithelial_cells)
  cat("Malignant fraction of epithelial cells:", round(malignant_fraction * 100, 1), "%\n")
}

# Save results
cat("\nSaving results...\n")
write.csv(predictions, file.path(out_dir, "copykat_predictions.csv"), row.names = TRUE)
saveRDS(ck_result, file.path(out_dir, "copykat_full_results.rds"))

# Save a summary
summary_df <- data.frame(
  total_cells = ncol(tumor_combined),
  epithelial_cells = length(epithelial_cells),
  analyzed_cells = length(analyzed_cells),
  malignant_cells = malignant_count,
  normal_epithelial = normal_epi_count,
  tumor_cells = tumor_count,
  malignant_fraction = if(length(epithelial_cells) > 0) malignant_count / length(epithelial_cells) else 0
)
write.csv(summary_df, file.path(out_dir, "copykat_summary.csv"), row.names = FALSE)

# Clean up memory
rm(epi_dense); gc()

cat("✓ CopyKAT analysis complete!\n")
cat("Results saved to:", out_dir, "\n")
cat("Key metadata added to Seurat object:\n")
cat("  - copykat_pred: aneuploid/diploid classification\n")
cat("  - copykat_confidence: probability of being aneuploid\n")
cat("  - malignant_call: Tumor_cells/Normal_ductal for epithelial cells\n")

In [ ]:
%%R
# SAVE ALL COPYKAT RESULTS
cat("=== SAVING COPYKAT RESULTS ===\n")

# Create directories
base_dir <- "/content/drive/MyDrive/BayesPrism"
copykat_dir <- file.path(base_dir, "copykat_results")
checkpoints_dir <- file.path(base_dir, "copykat_checkpoints")

dir.create(copykat_dir, showWarnings = FALSE, recursive = TRUE)
dir.create(checkpoints_dir, showWarnings = FALSE, recursive = TRUE)

cat("Saving to Google Drive path:", copykat_dir, "\n")

# SAVE MAIN COPYKAT RESULTS

if(exists("ck_result") && !is.null(ck_result)) {

  # 1. Save complete CopyKAT object
  saveRDS(ck_result, file.path(copykat_dir, "copykat_complete_results.rds"))
  cat("✓ Saved complete CopyKAT results object\n")

  # 2. Save predictions CSV
  predictions <- ck_result$prediction
  write.csv(predictions, file.path(copykat_dir, "copykat_predictions.csv"), row.names = TRUE)
  cat("✓ Saved predictions CSV (", nrow(predictions), "cells )\n")

  # 3. Save CNV matrix if available
  if("CNVmat" %in% names(ck_result)) {
    write.csv(ck_result$CNVmat, file.path(copykat_dir, "copykat_CNV_matrix.csv"))
    cat("✓ Saved CNV matrix\n")
  }

  # 4. Save distance matrix if available
  if("distance" %in% names(ck_result)) {
    saveRDS(ck_result$distance, file.path(copykat_dir, "copykat_distance_matrix.rds"))
    cat("✓ Saved distance matrix\n")
  }

} else {
  stop("CopyKAT results (ck_result) not found!")
}

# SAVE SEURAT OBJECT WITH COPYKAT RESULTS

if(exists("tumor_combined")) {
  # Save updated Seurat object with CopyKAT metadata
  saveRDS(tumor_combined, file.path(checkpoints_dir, "tumor_combined_with_copykat.rds"))
  cat("✓ Saved Seurat object with CopyKAT results\n")

  # Save metadata as CSV for easy access
  metadata_df <- tumor_combined@meta.data
  write.csv(metadata_df, file.path(copykat_dir, "seurat_metadata_with_copykat.csv"))
  cat("✓ Saved Seurat metadata with CopyKAT annotations\n")
} else {
  cat("⚠️  tumor_combined object not found\n")
}

# SAVE EPITHELIAL CELLS AND PROCESSED DATA

if(exists("epithelial_cells")) {
  saveRDS(epithelial_cells, file.path(checkpoints_dir, "epithelial_cells_used.rds"))

  # Save as CSV too
  epithelial_df <- data.frame(
    cell_barcode = epithelial_cells,
    sample_id = tumor_combined$sample_id[epithelial_cells],
    copykat_pred = tumor_combined$copykat_pred[epithelial_cells],
    malignant_call = tumor_combined$malignant_call[epithelial_cells]
  )
  write.csv(epithelial_df, file.path(copykat_dir, "epithelial_cells_analysis.csv"), row.names = FALSE)
  cat("✓ Saved epithelial cells data (", length(epithelial_cells), "cells )\n")
}

# CREATE SUMMARY STATISTICS

cat("Creating summary statistics...\n")

# Basic summary
summary_stats <- data.frame(
  metric = c("total_cells_dataset", "epithelial_cells_identified", "cells_analyzed_copykat",
             "aneuploid_cells", "diploid_cells", "tumor_cells", "normal_epithelial"),
  count = c(
    ncol(tumor_combined),
    length(epithelial_cells),
    nrow(predictions),
    sum(predictions$copykat.pred == "aneuploid"),
    sum(predictions$copykat.pred == "diploid"),
    sum(tumor_combined$malignant_call == "Tumor_cells", na.rm = TRUE),
    sum(tumor_combined$malignant_call == "Normal_ductal", na.rm = TRUE)
  ),
  percentage = c(
    100,
    round(length(epithelial_cells) / ncol(tumor_combined) * 100, 1),
    round(nrow(predictions) / ncol(tumor_combined) * 100, 1),
    round(sum(predictions$copykat.pred == "aneuploid") / nrow(predictions) * 100, 1),
    round(sum(predictions$copykat.pred == "diploid") / nrow(predictions) * 100, 1),
    round(sum(tumor_combined$malignant_call == "Tumor_cells", na.rm = TRUE) / ncol(tumor_combined) * 100, 1),
    round(sum(tumor_combined$malignant_call == "Normal_ductal", na.rm = TRUE) / ncol(tumor_combined) * 100, 1)
  )
)

write.csv(summary_stats, file.path(copykat_dir, "copykat_summary_statistics.csv"), row.names = FALSE)
cat("✓ Saved summary statistics\n")

# Per-sample summary
if(exists("tumor_combined") && "sample_id" %in% colnames(tumor_combined@meta.data)) {
  sample_summary <- tumor_combined@meta.data %>%
    group_by(sample_id) %>%
    summarise(
      total_cells = n(),
      epithelial_cells = sum(!is.na(malignant_call)),
      tumor_cells = sum(malignant_call == "Tumor_cells", na.rm = TRUE),
      normal_epithelial = sum(malignant_call == "Normal_ductal", na.rm = TRUE),
      aneuploid_cells = sum(copykat_pred == "aneuploid"),
      diploid_cells = sum(copykat_pred == "diploid"),
      tumor_fraction = round(sum(malignant_call == "Tumor_cells", na.rm = TRUE) / sum(!is.na(malignant_call)) * 100, 1)
    ) %>%
    arrange(desc(tumor_fraction))

  write.csv(sample_summary, file.path(copykat_dir, "per_sample_summary.csv"), row.names = FALSE)
  cat("✓ Saved per-sample summary\n")
}

# SAVE SESSION INFORMATION

session_info <- list(
  analysis_date = Sys.Date(),
  analysis_time = Sys.time(),
  seurat_version = as.character(packageVersion("Seurat")),
  copykat_version = as.character(packageVersion("copykat")),
  r_version = R.version.string,
  total_runtime = "CopyKAT completed",
  google_drive_path = copykat_dir,
  files_saved = list.files(copykat_dir)
)

saveRDS(session_info, file.path(copykat_dir, "session_info.rds"))

# Save as readable text too
session_text <- capture.output({
  cat("CopyKAT Analysis Session Information\n")
  cat("=====================================\n")
  cat("Date:", as.character(Sys.Date()), "\n")
  cat("Time:", as.character(Sys.time()), "\n")
  cat("Seurat version:", as.character(packageVersion("Seurat")), "\n")
  cat("CopyKAT version:", as.character(packageVersion("copykat")), "\n")
  cat("R version:", R.version.string, "\n")
  cat("Google Drive path:", copykat_dir, "\n")
  cat("\nAnalysis Summary:\n")
  print(summary_stats)
})

writeLines(session_text, file.path(copykat_dir, "session_info.txt"))
cat("✓ Saved session information\n")

# FINAL SUMMARY

cat("\n ALL COPYKAT RESULTS SAVED TO GOOGLE DRIVE!\n")
cat("="*60, "\n")
cat(" Main results folder:", copykat_dir, "\n")
cat(" Checkpoints folder:", checkpoints_dir, "\n")

cat("\n FILES SAVED:\n")

# List main results
cat("\nMain Results:\n")
result_files <- list.files(copykat_dir)
for(f in result_files) {
  size_mb <- round(file.size(file.path(copykat_dir, f)) / 1024 / 1024, 1)
  cat(" " f, "(", size_mb, "MB )\n")
}

# List checkpoints
cat("\nCheckpoints:\n")
checkpoint_files <- list.files(checkpoints_dir)
for(f in checkpoint_files) {
  size_mb <- round(file.size(file.path(checkpoints_dir, f)) / 1024 / 1024, 1)
  cat("  ", f, "(", size_mb, "MB )\n")
}

# Key statistics
cat("\n KEY RESULTS:\n")
cat("  Total cells in dataset:", ncol(tumor_combined), "\n")
cat("  Epithelial cells analyzed:", length(epithelial_cells), "\n")
cat("  Malignant cells found:", sum(tumor_combined$copykat_pred == "aneuploid"), "\n")
cat("  Tumor cells:", sum(tumor_combined$malignant_call == "Tumor_cells", na.rm = TRUE), "\n")
cat("  Normal epithelial:", sum(tumor_combined$malignant_call == "Normal_ductal", na.rm = TRUE), "\n")

if(length(epithelial_cells) > 0) {
  malignant_pct <- round(sum(tumor_combined$copykat_pred == "aneuploid") / length(epithelial_cells) * 100, 1)
  cat("  Malignancy rate:", malignant_pct, "%\n")
}

cat("\n CopyKAT analysis complete and saved!\n")


In [ ]:
%%R
suppressPackageStartupMessages({ library(Seurat); library(dplyr) })
cat("=== Identifier audit between Seurat and CopyKAT files ===\n")

# 0) Inputs
stopifnot(exists("tumor_combined"))
audit_out <- "/content/drive/MyDrive/BayesPrism/copykat_results/identifier_audit.csv"

files_to_check <- c(
  "epithelial_cells_analysis.csv",
  "seurat_metadata_with_copykat.csv",
  "copykat_predictions.csv"
)
files_to_check <- file.path("/content/drive/MyDrive/BayesPrism/copykat_results", files_to_check)
files_to_check <- files_to_check[file.exists(files_to_check)]

if (!length(files_to_check)) {
  stop("No CopyKAT result files found in /content/drive/MyDrive/BayesPrism/copykat_results")
}

# 1) Helpers
extract_core <- function(x) {
  x <- as.character(x)
  x <- gsub("\\s+", "", x)
  x <- sub("-\\d+$", "", x)
  x <- sub("_\\d+$", "", x)
  x <- sub("\\.\\d+$", "", x)
  up <- toupper(x)
  m  <- regexpr("[ACGTN]{10,}", up)
  core <- ifelse(m > 0, substr(up, m, m + attr(m, "match.length") - 1), up)
  core <- sub("^.*[_-]", "", core)
  core
}

norm_names <- function(v) gsub("\\.", "_", v)

read_one <- function(fp) {
  # Try with rownames first
  df <- tryCatch(
    read.csv(fp, stringsAsFactors = FALSE, row.names = 1, check.names = FALSE),
    error = function(e) {
      read.csv(fp, stringsAsFactors = FALSE, check.names = FALSE)
    }
  )
  nm <- norm_names(colnames(df))
  colnames(df) <- nm

  # Choose barcode column
  if (!is.null(rownames(df)) && length(rownames(df)) && any(nchar(rownames(df)) > 0)) {
    barcode <- rownames(df)
  } else if ("cell_barcode" %in% nm) {
    barcode <- df$cell_barcode
  } else if ("barcode" %in% nm) {
    barcode <- df$barcode
  } else {
    barcode <- character(0)
  }

  # Choose sample_id column if present
  sample_col <- NA
  cand <- c("sample_id","sampleid","orig_ident","orig_ident_","orig_ident2")
  cand <- cand[cand %in% nm]
  if (length(cand)) sample_col <- cand[1]

  data.frame(
    barcode   = as.character(barcode),
    sample_id = if (!is.na(sample_col)) as.character(df[[sample_col]]) else NA_character_,
    stringsAsFactors = FALSE
  )
}

# 2) Seurat identifiers
seu_bc   <- colnames(tumor_combined)
seu_sid  <- if ("sample_id" %in% colnames(tumor_combined@meta.data)) as.character(tumor_combined$sample_id) else rep(NA_character_, length(seu_bc))
seu_core <- extract_core(seu_bc)

seu_df <- data.frame(barcode = seu_bc, sample_id = seu_sid, core = seu_core, stringsAsFactors = FALSE)
cat("Seurat cells:", nrow(seu_df), " | unique samples in Seurat:", length(unique(na.omit(seu_df$sample_id))), "\n")

# 3) Audit loop
summary_list <- list()

for (fp in files_to_check) {
  cat("\n--- Auditing:", basename(fp), "---\n")
  ext <- read_one(fp)
  ext <- ext[!is.na(ext$barcode) & nzchar(ext$barcode), , drop = FALSE]
  ext$core <- extract_core(ext$barcode)
  has_sample <- any(!is.na(ext$sample_id) & nzchar(ext$sample_id))
  cat("Rows read:", nrow(ext), " | unique samples in file:", if (has_sample) length(unique(na.omit(ext$sample_id))) else 0, "\n")

  # Exact barcode overlap
  exact_hits <- intersect(seu_df$barcode, ext$barcode)
  cat("Exact barcode overlap:", length(exact_hits), "\n")
  if (length(exact_hits)) cat("  e.g.:", paste(head(exact_hits, 5), collapse = ", "), "\n")

  # Core-only overlap
  core_hits <- intersect(seu_df$core, ext$core)
  cat("Core-only overlap:", length(core_hits), "\n")
  if (length(core_hits)) cat("  e.g.:", paste(head(core_hits, 5), collapse = ", "), "\n")

  # Sample+core overlap (only if both sides have samples)
  sc_hits_n <- 0
  example_sc <- character(0)
  if (has_sample && any(!is.na(seu_df$sample_id) & nzchar(seu_df$sample_id))) {
    key_seu <- paste(seu_df$sample_id, seu_df$core, sep="|")
    key_ext <- paste(ext$sample_id,     ext$core,     sep="|")
    sc_hits <- intersect(key_seu, key_ext)
    sc_hits_n <- length(sc_hits)
    if (sc_hits_n) example_sc <- head(sc_hits, 5)
  }
  cat("Sample+core overlap:", sc_hits_n, "\n")
  if (length(example_sc)) cat("  e.g.:", paste(example_sc, collapse = ", "), "\n")

  # Collect summary row
  summary_list[[basename(fp)]] <- data.frame(
    file = basename(fp),
    rows_in_file = nrow(ext),
    exact_barcode_overlap = length(exact_hits),
    core_only_overlap = length(core_hits),
    sample_core_overlap = sc_hits_n,
    has_sample_id_in_file = has_sample,
    stringsAsFactors = FALSE
  )

  # Show a few non-overlapping examples (helps spot prefixes)
  if (length(core_hits) == 0 && length(exact_hits) == 0) {
    cat("No overlaps. Example Seurat barcodes:\n  ", paste(head(seu_df$barcode, 3), collapse = ", "), "\n")
    cat("Example file barcodes:\n  ", paste(head(ext$barcode, 3), collapse = ", "), "\n")
  }
}

audit_tbl <- bind_rows(summary_list)
write.csv(audit_tbl, audit_out, row.names = FALSE)
cat("\n=== Summary written to:", audit_out, "===\n")
print(audit_tbl)


In [ ]:
%%R

# STEP 4.2 — Map CopyKAT predictions to Seurat barcodes (explicit columns)
#   - Uses 'cell_names' and 'copykat_pred' from copykat_predictions.csv
#   - Bridges core -> full barcodes via epithelial_cells_analysis.csv

suppressPackageStartupMessages({ library(Seurat); library(dplyr) })

cat("=== STEP 4.2: Map CopyKAT predictions to Seurat (explicit) ===\n")

ck_dir   <- "/content/drive/MyDrive/BayesPrism/copykat_results"
ck_pred  <- file.path(ck_dir, "copykat_predictions.csv")
epi_csv  <- file.path(ck_dir, "epithelial_cells_analysis.csv")

stopifnot(exists("tumor_combined"))
stopifnot(file.exists(ck_pred))
stopifnot(file.exists(epi_csv))

extract_core <- function(x) {
  x <- as.character(x)
  x <- gsub("\\s+", "", x)
  x <- sub("-\\d+$", "", x)
  x <- sub("_\\d+$", "", x)
  x <- sub("\\.\\d+$", "", x)
  up <- toupper(x)
  m  <- regexpr("[ACGTN]{10,}", up)
  core <- ifelse(m > 0, substr(up, m, m + attr(m, "match.length") - 1), up)
  core <- sub("^.*[_-]", "", core)
  core
}

# Seurat barcodes ↔ core
seu_bc <- colnames(tumor_combined)
seu_df <- data.frame(barcode = seu_bc,
                     core    = extract_core(seu_bc),
                     stringsAsFactors = FALSE)

# epithelial_cells_analysis.csv => map of full barcode ↔ core (dedup by core)
epi_df <- read.csv(epi_csv, stringsAsFactors = FALSE, check.names = FALSE)
if ("cell_barcode" %in% names(epi_df)) {
  epi_df$barcode <- as.character(epi_df$cell_barcode)
} else if ("barcode" %in% names(epi_df)) {
  epi_df$barcode <- as.character(epi_df$barcode)
} else if (!is.null(rownames(epi_df)) && all(nchar(rownames(epi_df)) > 0)) {
  epi_df$barcode <- rownames(epi_df)
} else stop("No barcode column in epithelial_cells_analysis.csv")

epi_df$core <- extract_core(epi_df$barcode)
dup_n <- sum(duplicated(epi_df$core))
if (dup_n > 0) cat("Duplicate cores in epithelial_cells_analysis.csv:", dup_n, "\n")
epi_df <- epi_df[!duplicated(epi_df$core), c("barcode","core")]
epi_df <- inner_join(epi_df, seu_df, by = c("barcode","core"))  # keep only barcodes present in Seurat

# CopyKAT predictions — DO NOT use row.names, the file has regular columns
raw_pred <- read.csv(ck_pred, stringsAsFactors = FALSE, check.names = FALSE)

# Standardize names (but keep originals too)
cn <- colnames(raw_pred)
cn_std <- gsub("[^A-Za-z0-9]+", "_", cn)
colnames(raw_pred) <- cn_std

# Expect (after standardization) exactly: cell_names, copykat_pred
if (!all(c("cell_names","copykat_pred") %in% colnames(raw_pred))) {
  stop("Expected columns 'cell_names' and 'copykat_pred' in copykat_predictions.csv; found: ",
       paste(colnames(raw_pred), collapse = ", "))
}

pred_tbl <- raw_pred[, c("cell_names","copykat_pred")]
colnames(pred_tbl) <- c("id", "pred")
pred_tbl$core <- extract_core(pred_tbl$id)
pred_tbl <- pred_tbl[, c("core","pred")]
pred_tbl <- distinct(pred_tbl)  # de-dup any repeats

cat("Pred rows read:", nrow(pred_tbl), " | unique cores:", length(unique(pred_tbl$core)), "\n")

# Bridge core -> Seurat barcode via epithelial map
map_tbl <- inner_join(pred_tbl, epi_df, by = "core")   # adds 'barcode' that is in this Seurat
cat("Mapped predictions onto Seurat barcodes:", nrow(map_tbl), "\n")

# Attach to Seurat object
tumor_combined$copykat_pred       <- NA_character_
tumor_combined$copykat_confidence <- NA_real_

if (nrow(map_tbl) > 0) {
  idx <- match(map_tbl$barcode, colnames(tumor_combined))
  tumor_combined$copykat_pred[idx] <- as.character(map_tbl$pred)
}

cat("copykat_pred summary (after mapping):\n")
print(table(tumor_combined$copykat_pred, useNA = "ifany"))

meta_out <- file.path(ck_dir, "seurat_metadata_corrected.csv")
write.csv(tumor_combined@meta.data, meta_out)
cat("Wrote corrected Seurat metadata to:", meta_out, "\n")


In [ ]:
%%R
# STEP 4.3 — finalize CopyKAT labels and recompute marker scores

suppressPackageStartupMessages({ library(Seurat); library(Matrix) })
cat("=== STEP 4.3: Clean CopyKAT labels + recompute scores ===\n")

stopifnot(exists("tumor_combined"))
stopifnot(exists("calculate_marker_score"))
stopifnot(exists("comprehensive_markers"))
stopifnot(exists("qc_markers"))

# 1) Normalize copykat_pred values
tumor_combined$copykat_pred <- as.character(tumor_combined$copykat_pred)
tumor_combined$copykat_pred[
  tumor_combined$copykat_pred %in% c("not.defined","not_defined","Not.Defined","NotDefined","", NA)
] <- NA

# 2) Get expression matrix for scoring
expr_data <- tryCatch(
  as.matrix(GetAssayData(tumor_combined, layer = "data")),
  error = function(e) as.matrix(GetAssayData(tumor_combined, slot = "data"))
)
cat("Expression matrix:", nrow(expr_data), "genes x", ncol(expr_data), "cells\n")

# 3) Recompute all Phase-1 (comprehensive) marker scores
comp_scores <- list()
avail_df <- data.frame(set = character(), found = integer(), total = integer())
for (nm in names(comprehensive_markers)) {
  genes <- comprehensive_markers[[nm]]
  comp_scores[[nm]] <- calculate_marker_score(genes, expr_data)
  avail_df <- rbind(avail_df, data.frame(
    set = nm,
    found = length(intersect(genes, rownames(expr_data))),
    total = length(genes)
  ))
}
print(avail_df)

# 4) Normal epithelial score for tumor-vs-normal gating
normal_epi_score <- calculate_marker_score(qc_markers$normal_epithelial, expr_data)

# 5)sanity check of CopyKAT labels present
ck_tbl <- table(tumor_combined$copykat_pred, useNA = "ifany")
print(ck_tbl)


In [ ]:
%%R

if (!exists("comprehensive_markers") || !is.list(comprehensive_markers)) comprehensive_markers <- list()

add_panel <- function(name, genes) {
  if (!(name %in% names(comprehensive_markers))) {
    comprehensive_markers[[name]] <- genes
  } else {
    comprehensive_markers[[name]] <- unique(c(comprehensive_markers[[name]], genes))
  }
}

add_panel("tcells_general",  c("CD3D","CD3E","CD3G","TRAC"))
add_panel("bcells_general",  c("MS4A1","CD79A","CD79B"))
add_panel("nk_general",      c("KLRD1","NKG7","GNLY","PRF1"))
add_panel("myeloid_general", c("LYZ","S100A8","S100A9","LST1","CTSS","CCR2"))
add_panel("dc_general",      c("CD1C","FCER1A","CLEC9A"))

cat("Added immune panels.\n")


In [ ]:
%%R

suppressPackageStartupMessages({ library(Matrix); library(Seurat) })
cat("Recomputing marker scores using sparse ops...\n")

# 1) Get expression without forcing dense
expr <- tryCatch(
  GetAssayData(tumor_combined, layer = "data"),
  error = function(e) GetAssayData(tumor_combined, slot = "data")
)
# Ensure a sparse Matrix
if (!inherits(expr, "dgCMatrix")) {
  if (is.matrix(expr)) {
    expr <- Matrix::Matrix(expr, sparse = TRUE)
  } else {
    expr <- as(expr, "dgCMatrix")
  }
}

# 2) Sparse-safe scorer
score_signature_sparse <- function(markers, mat) {
  if (length(markers) == 0) return(rep(0, ncol(mat)))
  idx <- match(markers, rownames(mat))


  miss <- is.na(idx)
  if (any(miss)) {
    rn_up <- toupper(rownames(mat)); mk_up <- toupper(markers[miss])
    idx2 <- match(mk_up, rn_up); idx[miss] <- idx2
  }
  miss <- is.na(idx)
  if (any(miss)) {
    rn_nv <- sub("\\..*$", "", rownames(mat))
    mk_nv <- sub("\\..*$", "", markers[miss])
    idx3  <- match(mk_nv, rn_nv); idx[miss] <- idx3
  }

  idx <- idx[!is.na(idx)]
  if (!length(idx)) return(rep(0, ncol(mat)))
  Matrix::colSums(mat[idx, , drop = FALSE]) / length(idx)
}

# 3) Build comp_scores (Phase 1 panels)
stopifnot(exists("comprehensive_markers"))
comp_scores <- lapply(comprehensive_markers, score_signature_sparse, mat = expr)

# 4) QC score
stopifnot(exists("qc_markers"))
normal_epi_score <- score_signature_sparse(qc_markers$normal_epithelial, expr)

# 5) quick coverage peek
coverage <- vapply(names(comprehensive_markers), function(nm)
  length(intersect(comprehensive_markers[[nm]], rownames(expr))), integer(1))
print(sort(coverage, decreasing = TRUE)[1:min(10, length(coverage))])

cat("Done (sparse scoring). No dense coercion.\n")


In [ ]:
%%R
# STEP 5 — ENHANCED PHASE 1 CLASSIFICATION (CopyKAT + sparse scores)
cat("=== ENHANCED PHASE 1 CLASSIFICATION ===\n")

stopifnot(exists("tumor_combined"))
stopifnot(exists("comp_scores"))
stopifnot(exists("qc_markers"))
stopifnot(exists("normal_epi_score"))

n_cells <- ncol(tumor_combined)
comprehensive_types <- rep("Unknown", n_cells)
names(comprehensive_types) <- colnames(tumor_combined)

# Safe getter
get_score <- function(nm) {
  if (nm %in% names(comp_scores)) comp_scores[[nm]] else rep(0, n_cells)
}

# Scores
epi_score       <- get_score("epithelial_ductal")   # alias of epithelial_general
basal_score     <- get_score("basal_like")
classical_score <- get_score("classical")

fibro_q_score   <- get_score("fibroblasts_quiescent")
mycaf_score     <- get_score("caf_myofibroblastic")
icaf_score      <- get_score("caf_inflammatory")
apcaf_score     <- get_score("caf_antigen_presenting")

endo_score      <- get_score("endothelial_blood")
lymph_endo_sc   <- get_score("endothelial_lymphatic")
pericyte_score  <- get_score("pericytes")
stellate_score  <- get_score("stellate_cells")

acinar_score    <- get_score("acinar_cells")
mast_score      <- get_score("mast_cells")

tcell_score     <- get_score("tcells_general")
bcell_score     <- get_score("bcells_general")
nk_score        <- get_score("nk_general")
myeloid_score   <- get_score("myeloid_general")
dc_score        <- get_score("dc_general")

# IMMUNE negative gate (label broadly as Immune; not subtyping here)
immune_score <- pmax(tcell_score, bcell_score, nk_score, myeloid_score, dc_score, na.rm = TRUE)
immune_idx <- which(immune_score > 0.20 & epi_score < 0.30)
if (length(immune_idx)) comprehensive_types[immune_idx] <- "Immune"

# TIER 1: Epithelial (CopyKAT + normal ductal score)
cat("TIER 1: Epithelial classification with CopyKAT...\n")
ck <- tumor_combined$copykat_pred
ck <- tolower(as.character(ck))
epi_idx <- which(epi_score > 0.20)

if (length(epi_idx)) {
  is_aneuploid <- ck[epi_idx] == "aneuploid"
  is_diploid   <- ck[epi_idx] == "diploid"
  normal_high  <- normal_epi_score[epi_idx] > 0.40

  malignant_idx <- epi_idx[is_aneuploid & !normal_high]
  normal_idx    <- epi_idx[is_diploid | normal_high]

  # Tumor subtype split
  if (length(malignant_idx)) {
    bsc <- basal_score[malignant_idx]
    clc <- classical_score[malignant_idx]
    basal_cells     <- malignant_idx[bsc > clc & bsc > 0.20]
    classical_cells <- malignant_idx[clc > bsc & clc > 0.20]
    remaining_malig <- setdiff(malignant_idx, c(basal_cells, classical_cells))

    if (length(basal_cells))     comprehensive_types[basal_cells]     <- "Tumor_basal"
    if (length(classical_cells)) comprehensive_types[classical_cells] <- "Tumor_classical"
    if (length(remaining_malig)) comprehensive_types[remaining_malig] <- "Tumor_cells"
  }

  if (length(normal_idx)) comprehensive_types[normal_idx] <- "Normal_ductal"
  cat("  Epithelial: Malignant =", length(malignant_idx), ", Normal =", length(normal_idx), "\n")
}

# TIER 2: Stromal (CAF spectrum) — only on non-epithelial/non-immune
cat("TIER 2: Stromal classification (CAF)...\n")
non_epi_non_imm <- setdiff(
  which(epi_score < 0.30 & immune_score < 0.20),
  which(comprehensive_types %in% c("Tumor_basal","Tumor_classical","Tumor_cells","Normal_ductal"))
)
stroma_core <- intersect(which(fibro_q_score > 0.15), non_epi_non_imm)

if (length(stroma_core)) {
  mycaf_cells <- stroma_core[mycaf_score[stroma_core] > 0.30]
  icaf_cells  <- setdiff(stroma_core, mycaf_cells)
  icaf_cells  <- icaf_cells[icaf_score[icaf_cells] > 0.30]
  apcaf_cells <- stroma_core[apcaf_score[stroma_core] > 0.30]

  if (length(mycaf_cells)) comprehensive_types[mycaf_cells] <- "myCAF"
  if (length(icaf_cells))  comprehensive_types[icaf_cells]  <- "iCAF"
  if (length(apcaf_cells)) comprehensive_types[apcaf_cells] <- "apCAF"

  remaining_stroma <- setdiff(stroma_core, c(mycaf_cells, icaf_cells, apcaf_cells))
  if (length(remaining_stroma)) comprehensive_types[remaining_stroma] <- "Fibroblasts"
}

# TIER 3: Vascular / mural / stellate (non-epithelial)
cat("TIER 3: Vascular/mural/stellate...\n")
non_epi <- which(epi_score < 0.30)

endo_cells <- intersect(which(endo_score > 0.35 | lymph_endo_sc > 0.35), non_epi)
if (length(endo_cells)) comprehensive_types[endo_cells] <- "Endothelial"

pericyte_cells <- intersect(which(pericyte_score > 0.35), non_epi)
if (length(pericyte_cells)) comprehensive_types[pericyte_cells] <- "Pericytes_SMC"

stellate_cells_idx <- intersect(which(stellate_score > 0.30), non_epi)
if (length(stellate_cells_idx)) comprehensive_types[stellate_cells_idx] <- "Stellate"

# TIER 4: Other non-immune
acinar_cells_idx <- which(acinar_score > 0.30 & epi_score < 0.25)
if (length(acinar_cells_idx)) comprehensive_types[acinar_cells_idx] <- "Acinar"

mast_cells_idx <- which(mast_score > 0.30 & immune_score < 0.20 & epi_score < 0.30)
if (length(mast_cells_idx)) comprehensive_types[mast_cells_idx] <- "Mast"

# Commit
tumor_combined$comprehensive_cell_type <- comprehensive_types

cat("\nEnhanced Phase 1 classification results:\n")
print(sort(table(comprehensive_types), decreasing = TRUE))

unknown_pct <- mean(comprehensive_types == "Unknown") * 100
cat("Unknown cells:", round(unknown_pct, 1), "%\n")


In [ ]:
%%R
# STEP 5B — AUTO-CALIBRATION & UNKNOWN SALVAGE (per-sample thresholds)
suppressPackageStartupMessages({ library(Seurat); library(Matrix) })
cat("=== STEP 5B: Auto-calibrate thresholds and re-label Unknown ===\n")

stopifnot(exists("tumor_combined"))
stopifnot(exists("comp_scores"))
stopifnot(exists("qc_markers"))

n_cells <- ncol(tumor_combined)
sid <- as.character(tumor_combined$sample_id)

# Safe getter
gs <- function(nm) if (nm %in% names(comp_scores)) comp_scores[[nm]] else rep(0, n_cells)

# Scores already computed in STEP 4.3
epi_score       <- gs("epithelial_ductal")
basal_score     <- gs("basal_like")
classical_score <- gs("classical")

fibro_q_score   <- gs("fibroblasts_quiescent")
mycaf_score     <- gs("caf_myofibroblastic")
icaf_score      <- gs("caf_inflammatory")
apcaf_score     <- gs("caf_antigen_presenting")

endo_score      <- gs("endothelial_blood")
lymph_endo_sc   <- gs("endothelial_lymphatic")
pericyte_score  <- gs("pericytes")
stellate_score  <- gs("stellate_cells")

acinar_score    <- gs("acinar_cells")
mast_score      <- gs("mast_cells")

tcell_score     <- gs("tcells_general")
bcell_score     <- gs("bcells_general")
nk_score        <- gs("nk_general")
myeloid_score   <- gs("myeloid_general")
dc_score        <- gs("dc_general")
immune_score    <- pmax(tcell_score,bcell_score,nk_score,myeloid_score,dc_score,na.rm=TRUE)

# Also score RBC/platelets quickly (not in comp_scores by default)
expr_data <- tryCatch(
  { as.matrix(GetAssayData(tumor_combined, layer = "data")) },
  error = function(e) as.matrix(GetAssayData(tumor_combined, slot  = "data"))
)
score_sig <- function(markers) {
  g <- intersect(markers, rownames(expr_data))
  if (!length(g)) return(rep(0, n_cells))
  if (length(g)==1) as.numeric(expr_data[g,])
  else colMeans(expr_data[g, , drop=FALSE])
}
rbc_score  <- score_sig(qc_markers$erythrocytes)
plat_score <- score_sig(qc_markers$platelets)

# Helper: per-sample quantile threshold (with a floor)
thr_by_sample <- function(v, s, q=0.70, floor=0.15) {
  t <- tapply(v, s, function(x) max(stats::quantile(x, q, na.rm=TRUE), floor))
  unname(t[match(s, names(t))])
}

epi_thr      <- thr_by_sample(epi_score, sid, q=0.65, floor=0.15)
imm_thr      <- thr_by_sample(immune_score, sid, q=0.65, floor=0.10)
fibro_thr    <- thr_by_sample(fibro_q_score, sid, q=0.60, floor=0.10)
endo_thr     <- thr_by_sample(endo_score, sid, q=0.70, floor=0.10)
lymph_endo_thr <- thr_by_sample(lymph_endo_sc, sid, q=0.70, floor=0.10)
pericyte_thr <- thr_by_sample(pericyte_score, sid, q=0.70, floor=0.10)
stellate_thr <- thr_by_sample(stellate_score, sid, q=0.65, floor=0.10)
acinar_thr   <- thr_by_sample(acinar_score, sid, q=0.70, floor=0.10)
mast_thr     <- thr_by_sample(mast_score, sid, q=0.70, floor=0.10)
rbc_thr      <- thr_by_sample(rbc_score, sid, q=0.90, floor=0.15)
plat_thr     <- thr_by_sample(plat_score, sid, q=0.90, floor=0.15)

# Work only on Unknown from STEP 5
lab <- as.character(tumor_combined$comprehensive_cell_type)
unk_idx <- which(lab == "Unknown")
cat("Unknown before salvage:", length(unk_idx), "\n")
if (length(unk_idx)) {
  # 1) High-confidence heme contaminants first
  lab[unk_idx[rbc_score[unk_idx]  > rbc_thr[unk_idx]]]  <- "Erythrocytes"
  lab[unk_idx[plat_score[unk_idx] > plat_thr[unk_idx]]] <- "Platelets"
  unk_idx <- which(lab == "Unknown")

  # 2) Immune (broad)
  imm_idx <- unk_idx[immune_score[unk_idx] > imm_thr[unk_idx] & epi_score[unk_idx] < epi_thr[unk_idx]]
  if (length(imm_idx)) lab[imm_idx] <- "Immune"
  unk_idx <- which(lab == "Unknown")

  # 3) Epithelial second pass (then tumor subtyping by basal/classical)
  epi2 <- unk_idx[epi_score[unk_idx] > epi_thr[unk_idx]]
  if (length(epi2)) {
    ck <- tolower(as.character(tumor_combined$copykat_pred[epi2]))
    normal_hi <- if (exists("normal_epi_score")) normal_epi_score[epi2] > thr_by_sample(normal_epi_score, sid, q=0.65, floor=0.10)[epi2] else rep(FALSE, length(epi2))
    mal <- epi2[ck == "aneuploid" & !normal_hi]
    nor <- epi2[ck == "diploid"   |  normal_hi]

    if (length(mal)) {
      b <- basal_score[mal]; c_ <- classical_score[mal]
      b_cells  <- mal[b > c_ & b > stats::quantile(b, 0.50, na.rm=TRUE)]
      c_cells  <- mal[c_ > b & c_ > stats::quantile(c_,0.50, na.rm=TRUE)]
      others   <- setdiff(mal, c(b_cells, c_cells))
      if (length(b_cells)) lab[b_cells] <- "Tumor_basal"
      if (length(c_cells)) lab[c_cells] <- "Tumor_classical"
      if (length(others))  lab[others]  <- "Tumor_cells"
    }
    if (length(nor)) lab[nor] <- "Normal_ductal"
  }
  unk_idx <- which(lab == "Unknown")

  # 4) Stromal families
  stroma_core <- unk_idx[fibro_q_score[unk_idx] > fibro_thr[unk_idx]]
  if (length(stroma_core)) {
    myc <- stroma_core[mycaf_score[stroma_core] > thr_by_sample(mycaf_score, sid, q=0.70, floor=0.10)[stroma_core]]
    icf <- setdiff(stroma_core, myc)
    icf <- icf[icaf_score[icf] > thr_by_sample(icaf_score, sid, q=0.70, floor=0.10)[icf]]
    apc <- stroma_core[apcaf_score[stroma_core] > thr_by_sample(apcaf_score, sid, q=0.70, floor=0.10)[stroma_core]]
    if (length(myc)) lab[myc] <- "myCAF"
    if (length(icf)) lab[icf] <- "iCAF"
    if (length(apc)) lab[apc] <- "apCAF"
    remaining <- setdiff(stroma_core, c(myc, icf, apc))
    if (length(remaining)) lab[remaining] <- "Fibroblasts"
  }
  unk_idx <- which(lab == "Unknown")

  # 5) Endothelial / Pericyte / Stellate
  e_idx <- unk_idx[(endo_score[unk_idx] > endo_thr[unk_idx]) | (lymph_endo_sc[unk_idx] > lymph_endo_thr[unk_idx])]
  if (length(e_idx)) lab[e_idx] <- "Endothelial"
  unk_idx <- which(lab == "Unknown")

  p_idx <- unk_idx[pericyte_score[unk_idx] > pericyte_thr[unk_idx]]
  if (length(p_idx)) lab[p_idx] <- "Pericytes_SMC"
  unk_idx <- which(lab == "Unknown")

  s_idx <- unk_idx[stellate_score[unk_idx] > stellate_thr[unk_idx]]
  if (length(s_idx)) lab[s_idx] <- "Stellate"
  unk_idx <- which(lab == "Unknown")

  # 6) Remaining specialized
  a_idx <- unk_idx[acinar_score[unk_idx] > acinar_thr[unk_idx] & epi_score[unk_idx] < epi_thr[unk_idx]]
  if (length(a_idx)) lab[a_idx] <- "Acinar"
  m_idx <- unk_idx[mast_score[unk_idx] > mast_thr[unk_idx] & immune_score[unk_idx] < imm_thr[unk_idx]]
  if (length(m_idx)) lab[m_idx] <- "Mast"
  unk_idx <- which(lab == "Unknown")

  # 7) Final argmax fallback using z-scores across main families
  if (length(unk_idx)) {
    Z <- function(v) { x <- (v - median(v,na.rm=TRUE))/mad(v,na.rm=TRUE); x[!is.finite(x)] <- 0; x }
    fam <- cbind(
      Epi = Z(epi_score[unk_idx]),
      Imm = Z(immune_score[unk_idx]),
      Fib = Z(fibro_q_score[unk_idx]),
      End = Z(endo_score[unk_idx]),
      Per = Z(pericyte_score[unk_idx]),
      Stel= Z(stellate_score[unk_idx]),
      Acn = Z(acinar_score[unk_idx]),
      Mas = Z(mast_score[unk_idx])
    )
    best <- max.col(fam, ties.method = "first")
    maxv <- apply(fam, 1, max)
    sec  <- apply(fam, 1, function(r) sort(r, decreasing=TRUE)[2])
    margin <- maxv - sec
    picks <- c("Epithelial_unassigned","Immune","Fibroblasts","Endothelial","Pericytes_SMC","Stellate","Acinar","Mast")
    assignable <- which(maxv > 1.0 & margin > 0.25)  # conservative
    if (length(assignable)) lab[unk_idx[assignable]] <- picks[best[assignable]]
  }
}

tumor_combined$comprehensive_cell_type <- lab
cat("Unknown after salvage:", sum(lab=="Unknown"), "\n")
cat("New distribution:\n")
print(sort(table(lab), decreasing=TRUE))


In [ ]:
%%R
# STEP 5C — Tighten Stellate/Pericyte/Fibro labels + targeted reassign

suppressPackageStartupMessages({ library(Seurat); library(Matrix) })
cat("=== STEP 5C: Stellate specificity filter and reassign ===\n")

stopifnot(exists("tumor_combined"), exists("comp_scores"), exists("comprehensive_markers"))

n_cells <- ncol(tumor_combined)
sid <- as.character(tumor_combined$sample_id)

# Sparse-friendly getter (already computed in STEP 4.3)
gs <- function(nm) if (nm %in% names(comp_scores)) comp_scores[[nm]] else rep(0, n_cells)

# Key scores
fibro_q_score   <- gs("fibroblasts_quiescent")
mycaf_score     <- gs("caf_myofibroblastic")
icaf_score      <- gs("caf_inflammatory")
apcaf_score     <- gs("caf_antigen_presenting")
pericyte_score  <- gs("pericytes")
stellate_score  <- gs("stellate_cells")

# A stricter stellate signature (no VIM/COL1A1/ACTA2):
stellate_strict_markers <- c("GFAP","RGS5","PDGFRB","SPARC","DES","NOTCH3","CSPG4","ABCC9")
expr_m <- tryCatch(GetAssayData(tumor_combined, layer = "data"),
                   error = function(e) GetAssayData(tumor_combined, slot = "data"))
score_sig_sparse <- function(markers, X) {
  g <- intersect(markers, rownames(X))
  if (!length(g)) return(rep(0, ncol(X)))
  if (length(g) == 1) as.numeric(X[g, ])
  else Matrix::colMeans(X[g, , drop = FALSE])
}
stellate_strict_score <- score_sig_sparse(stellate_strict_markers, expr_m)

# Per-sample threshold helper
thr_by_sample <- function(v, s, q=0.70, floor=0.10) {
  tt <- tapply(v, s, function(x) max(stats::quantile(x, q, na.rm=TRUE), floor))
  unname(tt[match(s, names(tt))])
}

lab <- as.character(tumor_combined$comprehensive_cell_type)
pre_tab <- table(lab)
cat("Before tightening, Stellate =", pre_tab["Stellate"], "\n")

# Candidates that were labeled Stellate but fail strict support
stel_idx <- which(lab == "Stellate")
if (length(stel_idx)) {
  strict_thr <- thr_by_sample(stellate_strict_score, sid, q=0.60, floor=0.10)
  weak_stellate <- stel_idx[stellate_strict_score[stel_idx] < strict_thr[stel_idx]]

  if (length(weak_stellate)) {
    # Reassign by the best of: Pericytes_SMC, myCAF, iCAF, Fibroblasts
    M <- cbind(
      Per = pericyte_score[weak_stellate],
      myC = mycaf_score[weak_stellate],
      iCF = icaf_score[weak_stellate],
      Fib = fibro_q_score[weak_stellate]
    )
    pick <- max.col(M, ties.method = "first")
    maxv <- apply(M, 1, max)
    sec  <- apply(M, 1, function(r) sort(r, decreasing=TRUE)[2])
    margin <- maxv - sec

    # Only reassign if margin is decent
    ok <- which(margin > 0.03)
    if (length(ok)) {
      to_lab <- c("Pericytes_SMC","myCAF","iCAF","Fibroblasts")[pick[ok]]
      lab[weak_stellate[ok]] <- to_lab
    }
  }
}

tumor_combined$comprehensive_cell_type <- lab
post_tab <- sort(table(lab), decreasing = TRUE)
cat("After tightening:\n")
print(post_tab)


In [ ]:
%%R
# STEP 5D — Cluster-majority smoothing (assign Unknown by cluster)

suppressPackageStartupMessages({ library(Seurat) })
cat("=== STEP 5D: Cluster-majority smoothing ===\n")

stopifnot(exists("tumor_combined"))
if (is.null(Idents(tumor_combined))) {
  cat("No Idents set; using current clusters.\n")
}
clu <- Idents(tumor_combined)
lab <- as.character(tumor_combined$comprehensive_cell_type)

clusters <- levels(clu)
reassigned <- 0L

for (cl in clusters) {
  idx <- which(clu == cl)
  if (!length(idx)) next
  tab <- sort(table(lab[idx]), decreasing = TRUE)
  tab <- tab[names(tab) != "Unknown"]  # ignore Unknown when choosing a majority
  if (!length(tab)) next

  top_lab <- names(tab)[1]
  prop    <- as.numeric(tab[1]) / length(idx)

  # Only smooth if a cluster has a strong single label
  if (prop >= 0.65 && tab[1] >= 30) {
    unk <- idx[lab[idx] == "Unknown"]
    if (length(unk)) {
      lab[unk] <- top_lab
      reassigned <- reassigned + length(unk)
    }
  }
}

tumor_combined$comprehensive_cell_type <- lab
cat("Unknown reassigned by cluster majority:", reassigned, "\n")
print(sort(table(lab), decreasing = TRUE))

# Save an updated snapshot
if (!exists("ECM_RUN_DIR")) ECM_RUN_DIR <- "/content/drive/MyDrive/BayesPrism/two_phase_results_ecm"
dir.create(ECM_RUN_DIR, recursive = TRUE, showWarnings = FALSE)
write.csv(tumor_combined@meta.data,
          file.path(ECM_RUN_DIR, "seurat_metadata_with_comprehensive_types_autotuned_refined.csv"))


In [ ]:
%%R
# STEP 5E — Broad Immune gating (label Unknown → Immune; no subtyping)
#   - Uses sparse-friendly scoring
#   - Leaves epithelial/tumor/stroma/vascular labels untouched

suppressPackageStartupMessages({ library(Seurat); library(Matrix) })
cat("=== STEP 5E: Broad immune gating ===\n")

stopifnot(exists("tumor_combined"))

# Expression (sparse)
expr_m <- tryCatch(GetAssayData(tumor_combined, layer = "data"),
                   error = function(e) GetAssayData(tumor_combined, slot = "data"))

# Fallback scorer if score_signature_robust isn't in session
if (!exists("score_signature_robust")) {
  score_signature_robust <- function(markers, X) {
    g <- intersect(markers, rownames(X))
    if (!length(g)) return(rep(0, ncol(X)))
    if (length(g) == 1) as.numeric(X[g, ])
    else Matrix::colMeans(X[g, , drop = FALSE])
  }
}

# Minimal broad immune signatures (gating only)
sig_cd45   <- c("PTPRC")
sig_tcell  <- c("CD3D","CD3E","CD3G","TRAC")
sig_bcell  <- c("MS4A1","CD79A","CD79B","CD74")
sig_nk     <- c("KLRD1","NKG7","GNLY","PRF1")
sig_myelo  <- c("LYZ","S100A8","S100A9","LST1","CTSS","FCGR3A","CCR2")
sig_dc     <- c("FCER1A","CD1C","CLEC9A")

# Scores (sparse-friendly)
sc_cd45 <- score_signature_robust(sig_cd45, expr_m)
sc_t    <- score_signature_robust(sig_tcell, expr_m)
sc_b    <- score_signature_robust(sig_bcell, expr_m)
sc_nk   <- score_signature_robust(sig_nk, expr_m)
sc_my   <- score_signature_robust(sig_myelo, expr_m)
sc_dc   <- score_signature_robust(sig_dc, expr_m)

immune_gate <- pmax(sc_cd45, sc_t, sc_b, sc_nk, sc_my, sc_dc, na.rm = TRUE)

# Epithelial score for safety (use existing comp_scores if available)
epi_score <- if (exists("comp_scores") && "epithelial_ductal" %in% names(comp_scores)) {
  comp_scores$epithelial_ductal
} else {
  score_signature_robust(
    c("EPCAM","KRT8","KRT18","KRT19","KRT7","CDH1","CFTR","SOX9","SLC4A4"),
    expr_m
  )
}

# Per-sample adaptive threshold
sid <- if ("sample_id" %in% colnames(tumor_combined@meta.data))
         as.character(tumor_combined$sample_id) else rep("sample1", ncol(expr_m))
thr_by_sample <- function(v, s, q=0.70, floor=0.02) {
  tt <- tapply(v, s, function(x) max(stats::quantile(x, q, na.rm=TRUE), floor))
  unname(tt[match(s, names(tt))])
}
thr <- thr_by_sample(immune_gate, sid, q=0.70, floor=0.02)

# Only relabel cells that are currently Unknown and clearly non-epithelial
lab <- as.character(tumor_combined$comprehensive_cell_type)
candidates <- which(lab == "Unknown" & immune_gate > thr & epi_score < 0.25)

# Also require some CD45 support (robust to missing PTPRC)
cd45_ok <- if ("PTPRC" %in% rownames(expr_m)) {
  as.numeric(expr_m["PTPRC", ]) > 0
} else rep(TRUE, ncol(expr_m))
candidates <- intersect(candidates, which(cd45_ok))

lab[candidates] <- "Immune"
tumor_combined$comprehensive_cell_type <- lab

cat("Relabeled Unknown → Immune:", length(candidates), "cells\n")
print(sort(table(lab), decreasing = TRUE))

# Save updated metadata snapshot
if (!exists("ECM_RUN_DIR")) ECM_RUN_DIR <- "/content/drive/MyDrive/BayesPrism/two_phase_results_ecm"
dir.create(ECM_RUN_DIR, recursive = TRUE, showWarnings = FALSE)
write.csv(tumor_combined@meta.data,
          file.path(ECM_RUN_DIR, "seurat_metadata_with_comprehensive_types_autotuned_refined_immune.csv"))


In [ ]:
%%R
# STEP 5F — Salvage Unknowns: neural / Schwann / adipocytes (+ optional low-RNA)
suppressPackageStartupMessages({ library(Seurat); library(Matrix); library(dplyr) })
cat("=== STEP 5F: Salvage Unknowns (neural/Schwann/adipocytes) ===\n")

stopifnot(exists("tumor_combined"))

# Expression (sparse-friendly)
expr_m <- tryCatch(GetAssayData(tumor_combined, layer = "data"),
                   error = function(e) GetAssayData(tumor_combined, slot = "data"))

# Robust sparse-friendly scorer
score_signature_sparse <- function(markers, X) {
  if (length(markers) == 0) return(rep(0, ncol(X)))
  g <- intersect(markers, rownames(X))
  if (!length(g)) return(rep(0, ncol(X)))
  if (length(g) == 1) as.numeric(X[g, ])
  else if (inherits(X, "dgCMatrix") || inherits(X, "dgTMatrix")) Matrix::colMeans(X[g, , drop = FALSE])
  else colMeans(as.matrix(X[g, , drop = FALSE]))
}

# Per-sample adaptive threshold helper
thr_by_sample <- function(v, s, q = 0.70, floor = 0.02) {
  tt <- tapply(v, s, function(x) max(stats::quantile(x, q, na.rm = TRUE), floor))
  unname(tt[match(s, names(tt))])
}

# Pull sample IDs
sid <- if ("sample_id" %in% colnames(tumor_combined@meta.data))
         as.character(tumor_combined$sample_id) else rep("sample1", ncol(expr_m))
sid[is.na(sid)] <- "sample_unknown"

# Safety scores already in session? if not, recompute minimal gates
epi_score <- if (exists("comp_scores") && "epithelial_ductal" %in% names(comp_scores)) {
  comp_scores$epithelial_ductal
} else {
  score_signature_sparse(c("EPCAM","KRT8","KRT18","KRT19","KRT7","CDH1","CFTR","SOX9","SLC4A4"), expr_m)
}

immune_gate <- if (exists("immune_gate")) immune_gate else {
  sig_cd45 <- c("PTPRC")
  sig_t    <- c("CD3D","CD3E","CD3G","TRAC")
  sig_b    <- c("MS4A1","CD79A","CD79B","CD74")
  sig_nk   <- c("KLRD1","NKG7","GNLY","PRF1")
  sig_my   <- c("LYZ","S100A8","S100A9","LST1","CTSS","FCGR3A","CCR2")
  sig_dc   <- c("FCER1A","CD1C","CLEC9A")
  pmax(score_signature_sparse(sig_cd45, expr_m),
       score_signature_sparse(sig_t, expr_m),
       score_signature_sparse(sig_b, expr_m),
       score_signature_sparse(sig_nk, expr_m),
       score_signature_sparse(sig_my, expr_m),
       score_signature_sparse(sig_dc, expr_m), na.rm = TRUE)
}
thr_immune <- thr_by_sample(immune_gate, sid, q = 0.70, floor = 0.02)

# Marker sets (from earlier other_markers)
if (!exists("other_markers")) {
  other_markers <- list(
    neural_cells  = c("TUBB3","MAP2","SYP","CHGA","NCAM1","UCHL1"),
    schwann_cells = c("MPZ","PLP1","S100B","SOX10","NGFR","GFAP"),
    adipocytes    = c("ADIPOQ","LEP","PLIN1","FABP4","LPL")
  )
}

neural_sc   <- score_signature_sparse(other_markers$neural_cells,   expr_m)
schwann_sc  <- score_signature_sparse(other_markers$schwann_cells,  expr_m)
adipo_sc    <- score_signature_sparse(other_markers$adipocytes,     expr_m)

thr_neural  <- thr_by_sample(neural_sc,  sid, q = 0.70, floor = 0.01)
thr_schwann <- thr_by_sample(schwann_sc, sid, q = 0.70, floor = 0.01)
thr_adipo   <- thr_by_sample(adipo_sc,   sid, q = 0.75, floor = 0.01)  # adipocytes are sparse

lab <- as.character(tumor_combined$comprehensive_cell_type)
unknown_ix <- which(lab == "Unknown")

# Safety: do not steal likely epithelial or immune
safe_ix <- unknown_ix[ epi_score[unknown_ix] < 0.25 & immune_gate[unknown_ix] < thr_immune[unknown_ix] ]

# Assign in priority order to avoid overlaps (Schwann > Neural > Adipocytes)
pick_and_mark <- function(idx_pool, score, thr_vec, new_label) {
  cand <- idx_pool[ score[idx_pool] > thr_vec[idx_pool] ]
  if (length(cand)) lab[cand] <<- new_label
  setdiff(idx_pool, cand)
}

safe_ix <- pick_and_mark(safe_ix, schwann_sc, thr_schwann, "Schwann")
safe_ix <- pick_and_mark(safe_ix, neural_sc,  thr_neural,  "Neural")
safe_ix <- pick_and_mark(safe_ix, adipo_sc,   thr_adipo,   "Adipocytes")

tumor_combined$comprehensive_cell_type <- lab

cat("Assigned from Unknown -> Schwann/Neural/Adipocytes:\n")
print(table(lab)[c("Schwann","Neural","Adipocytes")], quote = FALSE)

cat("\nDistribution after STEP 5F:\n")
print(sort(table(lab), decreasing = TRUE))

# Optional: flag low-RNA cells as Low_quality (No overwrite existing labels)
if (!"Low_quality" %in% lab) {
  nfeat <- tumor_combined$nFeature_RNA
  low_cut <- max(200, stats::quantile(nfeat, 0.02, na.rm = TRUE))
  low_ix <- which(lab == "Unknown" & nfeat <= low_cut)
  if (length(low_ix)) {
    lab[low_ix] <- "Low_quality"
    tumor_combined$comprehensive_cell_type <- lab
    cat("\nMarked Low_quality (nFeature_RNA <= ", round(low_cut), "): ", length(low_ix), " cells\n", sep = "")
  }
}

# Save snapshot
if (!exists("ECM_RUN_DIR")) ECM_RUN_DIR <- "/content/drive/MyDrive/BayesPrism/two_phase_results_ecm"
dir.create(ECM_RUN_DIR, recursive = TRUE, showWarnings = FALSE)
write.csv(tumor_combined@meta.data,
          file.path(ECM_RUN_DIR, "seurat_metadata_with_comprehensive_types_step5f.csv"))
write.csv(as.data.frame(sort(table(lab), decreasing = TRUE)),
          file.path(ECM_RUN_DIR, "celltype_counts_step5f.csv"))


In [ ]:
%%R
# STEP 5G - Salvage Unknowns: endocrine / islet cells
suppressPackageStartupMessages({ library(Seurat); library(Matrix) })
cat("=== STEP 5G: Salvage Unknowns (endocrine/islet) ===\n")

stopifnot(exists("tumor_combined"))

expr_m <- tryCatch(GetAssayData(tumor_combined, layer = "data"),
                   error = function(e) GetAssayData(tumor_combined, slot = "data"))

# sparse-friendly scorer
score_signature_sparse <- function(markers, X) {
  if (length(markers) == 0) return(rep(0, ncol(X)))
  g <- intersect(markers, rownames(X))
  if (!length(g)) return(rep(0, ncol(X)))
  if (length(g) == 1) return(as.numeric(X[g, ]))
  if (inherits(X, "dgCMatrix") || inherits(X, "dgTMatrix")) {
    return(Matrix::colMeans(X[g, , drop = FALSE]))
  } else {
    return(colMeans(as.matrix(X[g, , drop = FALSE])))
  }
}

thr_by_sample <- function(v, s, q = 0.70, floor = 0.02) {
  tt <- tapply(v, s, function(x) max(stats::quantile(x, q, na.rm = TRUE), floor))
  unname(tt[match(s, names(tt))])
}

# sample ids
sid <- if ("sample_id" %in% colnames(tumor_combined@meta.data))
         as.character(tumor_combined$sample_id) else rep("sample1", ncol(expr_m))
sid[is.na(sid)] <- "sample_unknown"

# reuse gates if available; otherwise compute minimal ones
epi_score <- if (exists("comp_scores") && "epithelial_ductal" %in% names(comp_scores)) {
  comp_scores$epithelial_ductal
} else {
  score_signature_sparse(c("EPCAM","KRT8","KRT18","KRT19","KRT7","CDH1","CFTR","SOX9","SLC4A4"), expr_m)
}

immune_gate <- if (exists("immune_gate")) immune_gate else {
  pmax(
    score_signature_sparse(c("PTPRC"), expr_m),
    score_signature_sparse(c("CD3D","CD3E","CD3G","TRAC"), expr_m),
    score_signature_sparse(c("MS4A1","CD79A","CD79B","CD74"), expr_m),
    score_signature_sparse(c("KLRD1","NKG7","GNLY","PRF1"), expr_m),
    score_signature_sparse(c("LYZ","S100A8","S100A9","LST1","CTSS","FCGR3A","CCR2"), expr_m),
    score_signature_sparse(c("FCER1A","CD1C","CLEC9A"), expr_m),
    na.rm = TRUE
  )
}

thr_immune <- thr_by_sample(immune_gate, sid, q = 0.70, floor = 0.02)

# endocrine / islet signature
islet_sig <- if (exists("tumor_markers") && "normal_islet" %in% names(tumor_markers)) {
  tumor_markers$normal_islet
} else {
  c("INS","IAPP","PCSK1","CHGA","CHGB","SYP","GCG","SST","PPY","PDX1","PAX6","MAFA")
}

islet_sc  <- score_signature_sparse(islet_sig, expr_m)
thr_islet <- thr_by_sample(islet_sc, sid, q = 0.70, floor = 0.01)

lab <- as.character(tumor_combined$comprehensive_cell_type)
unknown_ix <- which(lab == "Unknown")

# assign safe Unknowns to islet
safe_ix <- unknown_ix[
  epi_score[unknown_ix]   < 0.25 &
  immune_gate[unknown_ix] < thr_immune[unknown_ix]
]
pick <- safe_ix[ islet_sc[safe_ix] > thr_islet[safe_ix] ]
if (length(pick)) lab[pick] <- "Islet_endocrine"

tumor_combined$comprehensive_cell_type <- lab
cat("Assigned Unknown -> Islet_endocrine:", length(pick), "\n")
print(sort(table(lab), decreasing = TRUE))

# save snapshot
if (!exists("ECM_RUN_DIR")) ECM_RUN_DIR <- "/content/drive/MyDrive/BayesPrism/two_phase_results_ecm"
dir.create(ECM_RUN_DIR, recursive = TRUE, showWarnings = FALSE)
write.csv(tumor_combined@meta.data,
          file.path(ECM_RUN_DIR, "seurat_metadata_with_comprehensive_types_step5g.csv"))
write.csv(as.data.frame(sort(table(lab), decreasing = TRUE)),
          file.path(ECM_RUN_DIR, "celltype_counts_step5g.csv"))


In [ ]:
%%R
# STEP 5H - Final stromal salvage (endothelial/pericyte/stellate/fibro)
suppressPackageStartupMessages({ library(Seurat); library(Matrix) })
cat("=== STEP 5H: Final stromal salvage ===\n")

stopifnot(exists("tumor_combined"))

# Expression matrix (sparse-friendly)
expr_m <- tryCatch(GetAssayData(tumor_combined, layer = "data"),
                   error = function(e) GetAssayData(tumor_combined, slot = "data"))

# Helpers
score_signature_sparse <- function(markers, X) {
  if (length(markers) == 0) return(rep(0, ncol(X)))
  g <- intersect(markers, rownames(X))
  if (!length(g)) return(rep(0, ncol(X)))
  if (length(g) == 1) return(as.numeric(X[g, ]))
  if (inherits(X, "dgCMatrix") || inherits(X, "dgTMatrix")) {
    return(Matrix::colMeans(X[g, , drop = FALSE]))
  } else {
    return(colMeans(as.matrix(X[g, , drop = FALSE])))
  }
}

thr_by_sample <- function(v, s, q = 0.65, floor = 0.01) {
  tt <- tapply(v, s, function(x) max(stats::quantile(x, q, na.rm = TRUE), floor))
  unname(tt[match(s, names(tt))])
}

# Sample ids
sid <- if ("sample_id" %in% colnames(tumor_combined@meta.data)) {
  as.character(tumor_combined$sample_id)
} else {
  rep("sample1", ncol(expr_m))
}
sid[is.na(sid)] <- "sample_unknown"

# Current labels
lab <- as.character(tumor_combined$comprehensive_cell_type)
lab[is.na(lab) | lab == ""] <- "Unknown"
unknown_ix <- which(lab == "Unknown")

# Reuse comp_scores when present, otherwise compute sparse scores on the fly
get_or_score <- function(name, fallback_genes) {
  if (exists("comp_scores") && name %in% names(comp_scores)) {
    return(comp_scores[[name]])
  } else {
    return(score_signature_sparse(fallback_genes, expr_m))
  }
}

epi_score    <- get_or_score("epithelial_ductal",
                  c("EPCAM","KRT8","KRT18","KRT19","KRT7","CDH1","CFTR","SOX9","SLC4A4"))
endo_sc      <- get_or_score("endothelial_blood",     c("PECAM1","VWF","CDH5","KDR","FLT1"))
lymph_endo   <- get_or_score("endothelial_lymphatic", c("PROX1","LYVE1","FLT4","PDPN","CCL21"))
peri_sc      <- get_or_score("pericytes",             c("RGS5","PDGFRB","CSPG4","MCAM","NOTCH3","ABCC9"))
stellate_sc  <- get_or_score("stellate_cells",        c("RGS5","PDGFRB","CSPG4","MCAM","NOTCH3","GFAP","SPARC","ACTA2"))
fibq_sc      <- get_or_score("fibroblasts_quiescent", c("DPT","DCN","LUM","COL1A1","COL1A2","COL3A1","BGN"))

# Broad immune gate to keep stromal salvage non-immune
immune_gate <- if (exists("immune_gate")) immune_gate else
  pmax(
    score_signature_sparse(c("PTPRC"), expr_m),
    score_signature_sparse(c("CD3D","CD3E","CD3G","TRAC"), expr_m),
    score_signature_sparse(c("MS4A1","CD79A","CD79B","CD74"), expr_m),
    score_signature_sparse(c("KLRD1","NKG7","GNLY","PRF1"), expr_m),
    score_signature_sparse(c("LYZ","S100A8","S100A9","LST1","CTSS","FCGR3A","CCR2"), expr_m),
    score_signature_sparse(c("FCER1A","CD1C","CLEC9A"), expr_m),
    na.rm = TRUE
  )

# Per-sample thresholds
thr_epi    <- thr_by_sample(epi_score,                 sid, q = 0.70, floor = 0.02)
thr_immune <- thr_by_sample(immune_gate,               sid, q = 0.70, floor = 0.02)
thr_endo   <- thr_by_sample(pmax(endo_sc, lymph_endo), sid, q = 0.65, floor = 0.01)
thr_peri   <- thr_by_sample(peri_sc,                   sid, q = 0.65, floor = 0.01)
thr_stel   <- thr_by_sample(stellate_sc,               sid, q = 0.65, floor = 0.01)
thr_fibq   <- thr_by_sample(fibq_sc,                   sid, q = 0.65, floor = 0.01)

# Candidate pool: non-epithelial and non-immune Unknowns
pool <- unknown_ix[
  epi_score[unknown_ix]   < thr_epi[unknown_ix] &
  immune_gate[unknown_ix] < thr_immune[unknown_ix]
]

# Priority order: Endothelial -> Pericytes_SMC -> Stellate -> Fibroblasts
picked <- integer(0)

pick <- pool[ pmax(endo_sc[pool], lymph_endo[pool], na.rm = TRUE) > thr_endo[pool] ]
if (length(pick)) { lab[pick] <- "Endothelial"; picked <- c(picked, pick); pool <- setdiff(pool, pick) }

pick <- pool[ peri_sc[pool] > thr_peri[pool] ]
if (length(pick)) { lab[pick] <- "Pericytes_SMC"; picked <- c(picked, pick); pool <- setdiff(pool, pick) }

pick <- pool[ stellate_sc[pool] > thr_stel[pool] ]
if (length(pick)) { lab[pick] <- "Stellate"; picked <- c(picked, pick); pool <- setdiff(pool, pick) }

pick <- pool[ fibq_sc[pool] > thr_fibq[pool] ]
if (length(pick)) { lab[pick] <- "Fibroblasts"; picked <- c(picked, pick); pool <- setdiff(pool, pick) }

tumor_combined$comprehensive_cell_type <- lab
cat("Unknown remaining after 5H:", sum(lab == "Unknown"), "\n")
print(sort(table(lab), decreasing = TRUE))

# Save snapshot
if (!exists("ECM_RUN_DIR")) ECM_RUN_DIR <- "/content/drive/MyDrive/BayesPrism/two_phase_results_ecm"
dir.create(ECM_RUN_DIR, recursive = TRUE, showWarnings = FALSE)
write.csv(tumor_combined@meta.data,
          file.path(ECM_RUN_DIR, "seurat_metadata_with_comprehensive_types_step5h.csv"))
write.csv(as.data.frame(sort(table(lab), decreasing = TRUE)),
          file.path(ECM_RUN_DIR, "celltype_counts_step5h.csv"))


In [ ]:
%%R
# STEP 5I — Nearest-centroid salvage for remaining stromal Unknowns
#   Targets: Endothelial / Pericytes_SMC / Stellate / Fibroblasts / myCAF / iCAF / apCAF
#   Uses only signature scores (sparse-friendly), no dense coercion.
suppressPackageStartupMessages({ library(Seurat); library(Matrix) })
cat("=== STEP 5I: Nearest-centroid salvage (stromal) ===\n")

stopifnot(exists("tumor_combined"))

# 1) Expression (sparse) and helpers
expr_m <- tryCatch(GetAssayData(tumor_combined, layer = "data"),
                   error = function(e) GetAssayData(tumor_combined, slot = "data"))

score_signature_sparse <- function(markers, X) {
  if (length(markers) == 0) return(rep(0, ncol(X)))
  g <- intersect(markers, rownames(X))
  if (!length(g)) return(rep(0, ncol(X)))
  if (length(g) == 1) return(as.numeric(X[g, ]))
  if (inherits(X, "dgCMatrix") || inherits(X, "dgTMatrix")) Matrix::colMeans(X[g, , drop = FALSE])
  else colMeans(as.matrix(X[g, , drop = FALSE]))
}

# 2) Define feature set (stromal signatures) and get scores
features <- c("endothelial_blood","endothelial_lymphatic",
              "pericytes","stellate_cells",
              "fibroblasts_quiescent","caf_myofibroblastic",
              "caf_inflammatory","caf_antigen_presenting")

get_scores <- function(sig_name) {
  if (exists("comp_scores") && sig_name %in% names(comp_scores)) {
    return(comp_scores[[sig_name]])
  }
  # fallback: compute from comprehensive_markers
  stopifnot(exists("comprehensive_markers"))
  score_signature_sparse(comprehensive_markers[[sig_name]], expr_m)
}

feat_mat <- do.call(rbind, lapply(features, get_scores))  # rows = features, cols = cells
rownames(feat_mat) <- features

# 3) Candidate labels and training pool (high-confidence non-Unknown)
cand_labels <- c("Endothelial","Pericytes_SMC","Stellate","Fibroblasts","myCAF","iCAF","apCAF")
lab <- as.character(tumor_combined$comprehensive_cell_type)
lab[is.na(lab) | lab == ""] <- "Unknown"

train_idx <- which(lab %in% cand_labels)
unknown_idx <- which(lab == "Unknown")
cat("Training cells:", length(train_idx), " | Unknown to classify:", length(unknown_idx), "\n")

# 4) Build centroids (mean feature vector per label)
build_centroid <- function(lbl) {
  ix <- which(lab == lbl)
  if (length(ix) < 50) return(rep(NA_real_, nrow(feat_mat)))  # need enough cells
  rowMeans(feat_mat[, ix, drop = FALSE])
}
C <- sapply(cand_labels, build_centroid)
colnames(C) <- cand_labels
# Drop labels with NA centroids
keep_lbl <- which(colSums(is.na(C)) == 0)
C <- C[, keep_lbl, drop = FALSE]
cand_use <- colnames(C)
cat("Centroids built for:", paste(cand_use, collapse = ", "), "\n")

if (length(cand_use) > 0 && length(unknown_idx) > 0) {
  # 5) Cosine similarity between Unknowns and centroids
  U <- feat_mat[, unknown_idx, drop = FALSE]     # F x Nunknown
  # normalize columns
  colnorm <- function(M) {
    d <- sqrt(colSums(M^2)); d[d == 0] <- 1
    sweep(M, 2, d, "/")
  }
  U_n <- colnorm(U)
  C_n <- colnorm(C)

  # sim = t(U_n) %*% C_n  -> Nunknown x K
  sim <- t(U_n) %*% C_n

  # 6) Assign with thresholds
  sim_thr <- 0.25
  margin  <- 0.05
  best_k  <- max.col(sim, ties.method = "first")
  best    <- sim[cbind(seq_len(nrow(sim)), best_k)]
  # runner-up for margin
  sim_sorted <- t(apply(sim, 1, function(x) sort(x, decreasing = TRUE)))
  second <- if (ncol(sim_sorted) >= 2) sim_sorted[,2] else rep(0, nrow(sim_sorted))
  ok <- which(best >= sim_thr & (best - second) >= margin)

  # 7) Consistency gate (keep non-epi, non-immune)
  #    Recompute quick broad gates (sparse-friendly)
  epi_gate <- score_signature_sparse(
    c("EPCAM","KRT8","KRT18","KRT19","KRT7","CDH1","CFTR","SOX9","SLC4A4"), expr_m)
  immune_gate <- pmax(
    score_signature_sparse(c("PTPRC"), expr_m),
    score_signature_sparse(c("CD3D","CD3E","CD3G","TRAC"), expr_m),
    score_signature_sparse(c("MS4A1","CD79A","CD79B","CD74"), expr_m),
    score_signature_sparse(c("KLRD1","NKG7","GNLY","PRF1"), expr_m),
    score_signature_sparse(c("LYZ","S100A8","S100A9","LST1","CTSS","FCGR3A","CCR2"), expr_m),
    score_signature_sparse(c("FCER1A","CD1C","CLEC9A"), expr_m),
    na.rm = TRUE
  )
  # per-sample dynamic thresholds
  sid <- if ("sample_id" %in% colnames(tumor_combined@meta.data)) as.character(tumor_combined$sample_id) else rep("sample1", ncol(expr_m))
  sid[is.na(sid)] <- "sample_unknown"
  thr_by_sample <- function(v, s, q = 0.70, floor = 0.02) {
    tt <- tapply(v, s, function(x) max(stats::quantile(x, q, na.rm = TRUE), floor))
    unname(tt[match(s, names(tt))])
  }
  thr_epi    <- thr_by_sample(epi_gate,    sid, q = 0.70, floor = 0.02)
  thr_immune <- thr_by_sample(immune_gate, sid, q = 0.70, floor = 0.02)

  ok_idx <- unknown_idx[ok]
  keep_consistent <- which(epi_gate[ok_idx] < thr_epi[ok_idx] &
                           immune_gate[ok_idx] < thr_immune[ok_idx])

  if (length(keep_consistent)) {
    new_lab <- cand_use[best_k[ok][keep_consistent]]
    lab[ ok_idx[keep_consistent] ] <- new_lab
  }
}

tumor_combined$comprehensive_cell_type <- lab
cat("Unknown remaining after 5I:", sum(lab == "Unknown"), "\n")
print(sort(table(lab), decreasing = TRUE))


In [ ]:
%%R
# STEP 5J — Light cluster-majority smoothing (on Unknown only)
#   Reassign Unknowns to cluster majority if majority >= 0.65 and cluster size >= 50
cat("=== STEP 5J: Cluster majority smoothing (final) ===\n")
stopifnot(exists("tumor_combined"))

lab <- as.character(tumor_combined$comprehensive_cell_type)
lab[is.na(lab) | lab == ""] <- "Unknown"
clu <- if ("seurat_clusters" %in% colnames(tumor_combined@meta.data)) tumor_combined$seurat_clusters else Idents(tumor_combined)
clu <- as.character(clu)

reassigned <- 0L
for (k in unique(clu)) {
  ix <- which(clu == k)
  if (length(ix) < 50) next
  tabk <- table(lab[ix])
  maj  <- names(tabk)[which.max(tabk)]
  pmaj <- max(tabk) / sum(tabk)
  if (maj != "Unknown" && pmaj >= 0.65) {
    uix <- ix[lab[ix] == "Unknown"]
    lab[uix] <- maj
    reassigned <- reassigned + length(uix)
  }
}
cat("Unknown reassigned by cluster majority:", reassigned, "\n")
tumor_combined$comprehensive_cell_type <- lab
print(sort(table(lab), decreasing = TRUE))


In [ ]:
%%R
# STEP 5L — Per-sample centroid salvage (stromal)
suppressPackageStartupMessages({ library(Seurat); library(Matrix) })
cat("=== STEP 5L: Per-sample centroid salvage (stromal) ===\n")

stopifnot(exists("tumor_combined"))

# Sparse-friendly scorer
score_signature_sparse <- function(markers, X) {
  if (length(markers) == 0) return(rep(0, ncol(X)))
  g <- intersect(markers, rownames(X))
  if (!length(g)) return(rep(0, ncol(X)))
  if (length(g) == 1) return(as.numeric(X[g, ]))
  if (inherits(X, "dgCMatrix") || inherits(X, "dgTMatrix")) Matrix::colMeans(X[g, , drop = FALSE])
  else colMeans(as.matrix(X[g, , drop = FALSE]))
}

# Expression (sparse) + IDs
expr_m <- tryCatch(GetAssayData(tumor_combined, layer = "data"),
                   error = function(e) GetAssayData(tumor_combined, slot = "data"))
sid <- if ("sample_id" %in% colnames(tumor_combined@meta.data))
         as.character(tumor_combined$sample_id) else rep("sample1", ncol(expr_m))
sid[is.na(sid)] <- "sample_unknown"

# Labels
lab <- as.character(tumor_combined$comprehensive_cell_type)
lab[is.na(lab) | lab == ""] <- "Unknown"

# Feature panels
panels <- list(
  endothelial_blood      = c("PECAM1","VWF","CDH5","KDR","FLT1","ESM1","PLVAP"),
  endothelial_lymphatic  = c("PROX1","LYVE1","FLT4","PDPN","CCL21"),
  pericytes              = c("RGS5","PDGFRB","CSPG4","MCAM","NOTCH3","ABCC9"),
  stellate_cells         = c("GFAP","SPARC","PDGFRB","VIM","DES"),
  fibroblasts_quiescent  = c("DPT","DCN","LUM","BGN","COL1A1","COL1A2","COL3A1"),
  caf_myofibroblastic    = c("ACTA2","TAGLN","MYL9","MYH11","CNN1","MYLK","POSTN","MMP11"),
  caf_inflammatory       = c("IL6","CXCL12","CXCL14","CCL2","PDGFRA","CFD","PLA2G2A"),
  caf_antigen_presenting = c("HLA-DRA","HLA-DPA1","HLA-DPB1","CD74","CIITA")
)

get_scores <- function(gset) {
  if (exists("comp_scores") && gset %in% names(comp_scores)) return(comp_scores[[gset]])
  score_signature_sparse(panels[[gset]], expr_m)
}
feat_names <- names(panels)
feat_mat <- do.call(rbind, lapply(feat_names, get_scores))
rownames(feat_mat) <- feat_names

cand_labels <- c("Endothelial","Pericytes_SMC","Stellate","Fibroblasts","myCAF","iCAF","apCAF")
label_to_feats <- list(
  Endothelial    = c("endothelial_blood","endothelial_lymphatic"),
  Pericytes_SMC  = c("pericytes"),
  Stellate       = c("stellate_cells"),
  Fibroblasts    = c("fibroblasts_quiescent"),
  myCAF          = c("caf_myofibroblastic"),
  iCAF          = c("caf_inflammatory"),
  apCAF          = c("caf_antigen_presenting")
)

# Combine feature rows per label -> Lmat (rows = labels, cols = cells)
label_rows <- lapply(cand_labels, function(lbl) {
  rows <- label_to_feats[[lbl]]
  if (length(rows) == 1) feat_mat[rows, , drop = FALSE] else colMeans(feat_mat[rows, , drop = FALSE])
})
Lmat <- do.call(rbind, label_rows)
rownames(Lmat) <- cand_labels
if (is.null(dim(Lmat))) Lmat <- matrix(Lmat, nrow = length(cand_labels), byrow = TRUE,
                                       dimnames = list(cand_labels, colnames(expr_m)))

# Cosine helper
colnorm <- function(M) { d <- sqrt(colSums(M^2)); d[d == 0] <- 1; sweep(M, 2, d, "/") }

unknown_idx <- which(lab == "Unknown")
reassigned <- 0L

for (s in unique(sid[unknown_idx])) {
  uix <- unknown_idx[sid[unknown_idx] == s]
  if (!length(uix)) next

  # training within sample
  train_idx <- which(sid == s & lab %in% cand_labels)
  if (length(train_idx) < 80) next

  # centroids from this sample
  C <- sapply(cand_labels, function(lbl) {
    ix <- which(sid == s & lab == lbl)
    if (length(ix) < 30) return(rep(NA_real_, nrow(Lmat)))
    rowMeans(Lmat[, ix, drop = FALSE])
  })
  if (is.null(dim(C))) C <- matrix(C, ncol = length(cand_labels),
                                   dimnames = list(rownames(Lmat), cand_labels))
  colnames(C) <- cand_labels
  keep <- colSums(is.na(C)) == 0
  if (!any(keep)) next
  C <- C[, keep, drop = FALSE]
  lbls <- colnames(C)

  # similarities for Unknown of this sample
  U   <- Lmat[, uix, drop = FALSE]
  sim <- t(colnorm(U)) %*% colnorm(C)    # Nunknown x Klabels

  best_k <- max.col(sim, ties.method = "first")
  best   <- sim[cbind(seq_len(nrow(sim)), best_k)]
  second <- apply(sim, 1, function(x) sort(x, decreasing = TRUE)[min(2, length(x))])

  thr   <- 0.28
  marg  <- 0.06
  ok    <- which(best >= thr & (best - second) >= marg)

  if (length(ok)) {
    lab[uix[ok]] <- lbls[best_k[ok]]
    reassigned <- reassigned + length(ok)
  }
}

tumor_combined$comprehensive_cell_type <- lab
cat("Reassigned in 5L:", reassigned, "\n")
cat("Unknown remaining:", sum(lab == "Unknown"), "\n")
print(sort(table(lab), decreasing = TRUE))


In [ ]:
%%R
# STEP 5M — Recluster Unknowns and auto-annotate by marker rules
#   Lightweight: variable features only; prints top markers per cluster
suppressPackageStartupMessages({ library(Seurat); library(dplyr) })
cat("=== STEP 5M: Unknown-only recluster + rule-based annotate ===\n")

lab <- as.character(tumor_combined$comprehensive_cell_type)
unk_cells <- colnames(tumor_combined)[lab == "Unknown"]
cat("Unknown cells to recluster:", length(unk_cells), "\n")
if (length(unk_cells) < 200) {
  cat("Too few Unknowns to recluster; skipping.\n")
} else {
  unk <- subset(tumor_combined, cells = unk_cells)

  DefaultAssay(unk) <- "RNA"
  unk <- NormalizeData(unk, verbose = FALSE)
  unk <- FindVariableFeatures(unk, selection.method = "vst", nfeatures = 2000, verbose = FALSE)
  unk <- ScaleData(unk, features = VariableFeatures(unk), verbose = FALSE)
  unk <- RunPCA(unk, npcs = 40, verbose = FALSE)
  set.seed(42)
  unk <- FindNeighbors(unk, dims = 1:30, verbose = FALSE)
  unk <- FindClusters(unk, resolution = 0.4, verbose = FALSE)
  unk <- RunUMAP(unk, dims = 1:30, verbose = FALSE)

  # Top markers (quick view)
  mk <- FindAllMarkers(unk, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.25)
  top5 <- mk %>% group_by(cluster) %>% top_n(n = 5, wt = avg_log2FC) %>% arrange(cluster, desc(avg_log2FC))
  print(top5 %>% select(cluster, gene, avg_log2FC, p_val_adj) %>% head(40))

  # Rule-based mapping using median signature scores per cluster
  exprU <- tryCatch(GetAssayData(unk, layer = "data"),
                    error = function(e) GetAssayData(unk, slot = "data"))
  med_sig <- function(genes) {
    g <- intersect(genes, rownames(exprU))
    if (!length(g)) return(rep(0, length(unique(Idents(unk)))))
    tapply(Matrix::colMeans(exprU[g, , drop = FALSE]), Idents(unk), median)
  }

  rules <- list(
    Endothelial   = c("PECAM1","VWF","CDH5","ESM1","PLVAP","KDR"),
    Pericytes_SMC = c("RGS5","PDGFRB","CSPG4","MCAM","NOTCH3","MYH11","TAGLN","ACTA2","CNN1"),
    Stellate      = c("GFAP","SPARC","VIM","PDGFRB","DES"),
    Fibroblasts   = c("DCN","LUM","BGN","COL1A1","COL1A2","COL3A1"),
    myCAF         = c("ACTA2","TAGLN","MYL9","MYH11","CNN1","POSTN","MMP11"),
    iCAF          = c("IL6","CXCL12","CXCL14","CCL2","PDGFRA"),
    apCAF         = c("HLA-DRA","HLA-DPA1","HLA-DPB1","CD74","CIITA")
  )

  clus <- levels(Idents(unk))
  score_mat <- sapply(names(rules), function(nm) med_sig(rules[[nm]]))
  score_mat <- as.matrix(score_mat)  # clusters x labels
  score_mat[is.na(score_mat)] <- 0

  best_lbl <- apply(score_mat, 1, function(x) names(x)[which.max(x)])
  best_val <- apply(score_mat, 1, max)
  # thresholds to avoid over-calling
  keep <- best_val >= quantile(best_val, 0.6, na.rm = TRUE)
  map <- setNames(ifelse(keep, best_lbl, NA), clus)

  # Apply to Unknowns
  new_lab <- as.character(Idents(unk))
  new_lab <- unname(map[new_lab])
  ok <- which(!is.na(new_lab))
  if (length(ok)) {
    tumor_combined$comprehensive_cell_type[unk_cells[ok]] <- new_lab[ok]
  }
  cat("Clusters auto-annotated:", sum(!is.na(map)), "of", length(map), "\n")
  cat("Remaining Unknown after 5M:",
      sum(tumor_combined$comprehensive_cell_type == "Unknown"), "\n")

  # Optional: save markers for manual QC
  out_dir <- if (exists("ECM_RUN_DIR")) ECM_RUN_DIR else "/content/drive/MyDrive/BayesPrism/two_phase_results_ecm"
  write.csv(mk, file.path(out_dir, "unknown_recluster_markers.csv"), row.names = FALSE)
}


In [ ]:
%%R
# STEP 5N (revised) — Rule-based finalize Unknowns with z-scored signatures
suppressPackageStartupMessages({ library(Seurat); library(Matrix); library(dplyr) })
cat("=== STEP 5N (revised): finalize Unknowns with z-scores ===\n")

stopifnot(exists("tumor_combined"))

# Current labels & Unknown set
lab <- as.character(tumor_combined$comprehensive_cell_type)
lab[is.na(lab) | lab == ""] <- "Unknown"
unknown_idx <- which(lab == "Unknown")
if (!length(unknown_idx)) { cat("No Unknown cells left.\n"); quit(save="no") }

unknown_sub <- subset(tumor_combined, cells = colnames(tumor_combined)[unknown_idx])
Idents(unknown_sub) <- if ("seurat_clusters" %in% colnames(unknown_sub@meta.data)) unknown_sub$seurat_clusters else 0

# Compact, non-overlapping meta-signatures
sig <- list(
  Immune_core   = c("PTPRC","CD3D","CD3E","CD3G","TRAC","MS4A1","CD79A","NKG7","GNLY","PRF1","LYZ","S100A8","S100A9","LST1","FCGR3B","CD14","HLA-DRA","HLA-DPA1","HLA-DPB1"),
  RBC           = c("HBA1","HBA2","HBB","ALAS2"),
  Platelets     = c("PPBP","PF4","ITGA2B","TUBB1"),
  Mast          = c("TPSAB1","CPA3","KIT"),
  Endothelial   = c("PECAM1","VWF","CDH5","KDR","FLT1","FLI1","ERG"),
  Pericyte_SMC  = c("RGS5","PDGFRB","CSPG4","MCAM","NOTCH3","ACTA2","MYH11","TAGLN","CNN1","MYLK"),
  Fibro_core    = c("COL1A1","COL1A2","COL3A1","DCN","LUM","BGN"),
  Stellate      = c("RGS5","NOTCH3","GFAP","SPARC","DES","ABCC9"),
  Acinar        = c("PRSS1","CPA1","CTRB1","CELA3A","AMY2A","PNLIP","CTRB2","CTRC"),
  Islet         = c("INS","GCG","SST","PPY","IAPP","CHGA","SYP"),
  Ductal        = c("EPCAM","KRT7","KRT8","KRT18","KRT19","MUC1","SOX9","SLC4A4","BICC1","GLIS3","SCTR","TFF3","CA2","ANXA4"),
  BasalSquam    = c("TP63","KRT5","KRT6A","KRT14","KRT13","DSC3","PKP1","SERPINB3","SERPINB4","PPL"),
  Schwann       = c("MPZ","PLP1","SOX10","S100B","NGFR","GFAP"),
  Neural        = c("TUBB3","MAP2","UCHL1","DCX","RBFOX3"),
  Adipocytes    = c("ADIPOQ","PLIN1","FABP4","LPL")
)

score_signature_sparse <- function(genes, M) {
  g <- intersect(genes, rownames(M))
  if (!length(g)) return(rep(0, ncol(M)))
  if (length(g) == 1) return(as.numeric(M[g, ]))
  if (inherits(M, "dgCMatrix") || inherits(M, "dgTMatrix")) Matrix::colMeans(M[g, , drop = FALSE])
  else colMeans(as.matrix(M[g, , drop = FALSE]))
}

Xu <- tryCatch(GetAssayData(unknown_sub, layer = "data"),
               error = function(e) GetAssayData(unknown_sub, slot = "data"))

# Per-cell scores in Unknown-only, then per-cluster means
cell_scores <- lapply(sig, score_signature_sparse, M = Xu)
sc_df <- as.data.frame(cell_scores)
sc_df$cluster <- Idents(unknown_sub)

clus_means <- sc_df %>%
  group_by(cluster) %>%
  summarise(across(all_of(names(sig)), \(x) mean(x, na.rm = TRUE))) %>%
  ungroup()

# Z-score each signature across clusters (relative dominance)
zdf <- clus_means
for (nm in names(sig)) {
  v <- zdf[[nm]]
  m <- mean(v, na.rm = TRUE); s <- sd(v, na.rm = TRUE)
  zdf[[nm]] <- if (is.na(s) || s == 0) rep(0, length(v)) else (v - m) / s
}

# CopyKAT aneuploid fraction per Unknown cluster (NA -> 0)
ck_vec <- as.character(tumor_combined$copykat_pred); ck_vec[is.na(ck_vec)] <- "NA"
ck_prop <- data.frame(cluster = Idents(unknown_sub),
                      ck = ck_vec[colnames(unknown_sub)]) %>%
  group_by(cluster) %>%
  summarise(prop_aneu = mean(ck == "aneuploid", na.rm = TRUE)) %>%
  mutate(prop_aneu = ifelse(is.na(prop_aneu), 0, prop_aneu)) %>%
  ungroup()

# Decision on z-scores (lowered thresholds; epithelial uses prop_aneu)
decide_label_z <- function(row, prop_aneu) {
  x <- as.numeric(row[names(sig)]); names(x) <- names(sig)
  x[is.na(x)] <- -Inf
  ord <- order(x, decreasing = TRUE)
  top <- names(x)[ord[1]]; topv <- x[ord[1]]
  secv <- x[ord[2]]; margin <- topv - secv
  pa <- ifelse(is.na(prop_aneu), 0, prop_aneu)

  if (top %in% c("BasalSquam","Ductal")) {
    if (pa >= 0.15) return(if (top == "BasalSquam") "Tumor_basal" else "Tumor_classical")
    return("Normal_ductal")
  }

  if (top=="Immune_core"  && topv>0.4 && margin>0.25) return("Immune")
  if (top=="RBC"          && topv>0.4 && margin>0.25) return("Erythrocytes")
  if (top=="Platelets"    && topv>0.4 && margin>0.25) return("Platelets")
  if (top=="Mast"         && topv>0.4 && margin>0.25) return("Mast")
  if (top=="Endothelial"  && topv>0.3 && margin>0.2)  return("Endothelial")
  if (top=="Pericyte_SMC" && topv>0.3 && margin>0.2)  return("Pericytes_SMC")
  if (top=="Fibro_core"   && topv>0.3 && margin>0.2)  return("Fibroblasts")
  if (top=="Stellate"     && topv>0.3 && margin>0.2)  return("Stellate")
  if (top=="Acinar"       && topv>0.3 && margin>0.2)  return("Acinar")
  if (top=="Islet"        && topv>0.3 && margin>0.2)  return("Islet_endocrine")
  if (top=="Schwann"      && topv>0.35&& margin>0.2)  return("Schwann")
  if (top=="Neural"       && topv>0.35&& margin>0.2)  return("Neural")
  if (top=="Adipocytes"   && topv>0.35&& margin>0.2)  return("Adipocytes")
  "Unknown"
}

zdf2 <- zdf %>%
  left_join(ck_prop, by = "cluster") %>%
  rowwise() %>%
  mutate(final_label = decide_label_z(c_across(all_of(names(sig))), prop_aneu)) %>%
  ungroup()

print(zdf2 %>% select(cluster, final_label, prop_aneu))

# Push labels back
new_lab_unknown <- setNames(rep("Unknown", ncol(unknown_sub)), colnames(unknown_sub))
for (k in seq_len(nrow(zdf2))) {
  cl <- as.character(zdf2$cluster[k])
  lab_k <- as.character(zdf2$final_label[k])
  if (lab_k != "Unknown") {
    idx <- which(Idents(unknown_sub) == cl)
    new_lab_unknown[idx] <- lab_k
  }
}

lab[colnames(unknown_sub)] <- new_lab_unknown
tumor_combined$comprehensive_cell_type <- lab

cat("Remaining Unknown after 5N (revised):", sum(lab == "Unknown"), "\n")
cat("Distribution after 5N (revised):\n")
print(sort(table(lab), decreasing = TRUE))


In [ ]:
%%R
# STEP 5P — CD45+ filtering for Unknowns + build ECM-only subset
suppressPackageStartupMessages({ library(Seurat); library(Matrix); library(dplyr) })
cat("=== STEP 5P: CD45+ filtering and ECM-only subset ===\n")

stopifnot(exists("tumor_combined"))
stopifnot(exists("negative_markers"))
stopifnot("immune_cd45" %in% names(negative_markers))

# Sparse-friendly scorer
score_signature_sparse <- function(genes, X) {
  g <- intersect(genes, rownames(X))
  if (!length(g)) return(rep(0, ncol(X)))
  if (length(g) == 1) return(as.numeric(X[g, ]))
  if (inherits(X, "dgCMatrix") || inherits(X, "dgTMatrix")) Matrix::colMeans(X[g, , drop = FALSE])
  else colMeans(as.matrix(X[g, , drop = FALSE]))
}

# Per-sample adaptive threshold
thr_by_sample <- function(v, s, q = 0.70, floor = 0.02) {
  tlist <- tapply(v, s, function(x) max(stats::quantile(x, q, na.rm = TRUE), floor))
  unname(tlist[match(s, names(tlist))])
}

# Expression matrix (sparse) and sample ids
expr_m <- tryCatch(GetAssayData(tumor_combined, layer = "data"),
                   error = function(e) GetAssayData(tumor_combined, slot = "data"))
sid <- if ("sample_id" %in% colnames(tumor_combined@meta.data)) {
  as.character(tumor_combined$sample_id)
} else rep("sample1", ncol(expr_m))
sid[is.na(sid)] <- "sample_unknown"

# 1) CD45/immune-core score for every cell
immune_genes <- unique(negative_markers$immune_cd45)
immune_score <- score_signature_sparse(immune_genes, expr_m)
names(immune_score) <- colnames(tumor_combined)

# Adaptive, per-sample CD45 threshold
imm_thr <- thr_by_sample(immune_score, sid, q = 0.70, floor = 0.02)
cd45_pos <- immune_score > imm_thr

# 2) Re-label Unknown → Immune if CD45+
lab <- as.character(tumor_combined$comprehensive_cell_type)
lab[is.na(lab) | lab == ""] <- "Unknown"

unknown_idx <- which(lab == "Unknown")
relabel_idx <- unknown_idx[cd45_pos[unknown_idx]]

if (length(relabel_idx)) {
  lab[relabel_idx] <- "Immune"
  cat("Relabeled Unknown → Immune (CD45+):", length(relabel_idx), "cells\n")
} else {
  cat("No Unknown cells exceeded CD45 threshold.\n")
}

tumor_combined$comprehensive_cell_type <- lab

cat("Distribution after CD45 relabel:\n")
print(sort(table(lab), decreasing = TRUE))

# 3) Build ECM-only subset:
#    - Exclude: Immune (CD45+), Erythrocytes, Platelets, Mast, Low_quality
#    - Also exclude any cells with CD45 score above threshold even if not labeled yet
exclude_types <- c("Immune","Erythrocytes","Platelets","Mast","Low_quality")
keep_ecm <- !(lab %in% exclude_types) & !cd45_pos

ecm_cells <- colnames(tumor_combined)[keep_ecm]
cat("ECM-only cells retained:", length(ecm_cells), "of", length(lab), "\n")

tumor_ecm <- subset(tumor_combined, cells = ecm_cells)

# Save outputs
ecm_dir <- "/content/drive/MyDrive/BayesPrism/two_phase_results_ecm"
dir.create(ecm_dir, showWarnings = FALSE, recursive = TRUE)

write.csv(tumor_combined@meta.data,
          file.path(ecm_dir, "seurat_metadata_with_cd45_filter.csv"))
saveRDS(tumor_ecm, file.path(ecm_dir, "tumor_ecm_subset.rds"))

# Quick summary tables
dist_all <- as.data.frame(sort(table(lab), decreasing = TRUE))
colnames(dist_all) <- c("cell_type","n")
write.csv(dist_all, file.path(ecm_dir, "celltype_distribution_after_cd45.csv"), row.names = FALSE)

cat("Wrote:\n",
    " - seurat_metadata_with_cd45_filter.csv\n",
    " - tumor_ecm_subset.rds\n",
    " - celltype_distribution_after_cd45.csv\n", sep = "")


In [ ]:
%%R
# STEP 5Q — ECM-only summaries
ecm_dir <- "/content/drive/MyDrive/BayesPrism/two_phase_results_ecm"

stopifnot(exists("tumor_ecm"))
lab_ecm <- as.character(tumor_ecm$comprehensive_cell_type)
tbl_ecm <- sort(table(lab_ecm), decreasing = TRUE)
print(tbl_ecm)

ecm_by_sample <- table(tumor_ecm$sample_id, lab_ecm)
write.csv(as.data.frame(tbl_ecm), file.path(ecm_dir, "ecm_celltype_counts.csv"))
write.csv(as.data.frame(ecm_by_sample), file.path(ecm_dir, "ecm_celltype_by_sample.csv"))


In [ ]:
%%R
# STEP 5R — Salvage residual Unknowns in ECM-only via signature max-z
suppressPackageStartupMessages({ library(Matrix) })

stopifnot(exists("tumor_ecm"), exists("stromal_markers"),
          exists("comprehensive_markers"), exists("qc_markers"))

# Get expression for the ECM subset
expr_ecm <- tryCatch(GetAssayData(tumor_ecm, layer = "data"),
                     error = function(e) GetAssayData(tumor_ecm, slot = "data"))

# Sparse-friendly scoring
score_sig <- function(genes, X) {
  g <- intersect(genes, rownames(X))
  if (!length(g)) return(rep(0, ncol(X)))
  if (length(g) == 1) return(as.numeric(X[g, ]))
  if (inherits(X, "dgCMatrix") || inherits(X, "dgTMatrix")) {
    Matrix::colMeans(X[g, , drop = FALSE])
  } else {
    colMeans(as.matrix(X[g, , drop = FALSE]))
  }
}

# Define panels
sig_panels <- list(
  myCAF       = stromal_markers$myCAF,
  iCAF        = stromal_markers$iCAF,
  apCAF       = stromal_markers$apCAF,
  Fibro       = comprehensive_markers$fibroblasts_quiescent,
  EndoBlood   = comprehensive_markers$endothelial_blood,
  EndoLymph   = comprehensive_markers$endothelial_lymphatic,
  PeriSMC     = comprehensive_markers$pericytes,
  Stellate    = comprehensive_markers$stellate_cells,
  Acinar      = comprehensive_markers$acinar_cells,
  NormalDuct  = qc_markers$normal_epithelial,
  TumorBasal  = comprehensive_markers$basal_like,
  TumorClass  = comprehensive_markers$classical
)

# Score each panel
score_list <- lapply(sig_panels, function(genes) score_sig(genes, expr_ecm))
sc_mat <- do.call(cbind, score_list)
colnames(sc_mat) <- names(sig_panels)

# Column-wise z-score (robust if a column is constant)
z_col <- function(v) {
  m <- mean(v, na.rm = TRUE); s <- stats::sd(v, na.rm = TRUE)
  if (!is.finite(s) || s == 0) return(rep(0, length(v)))
  (v - m) / s
}
sc_z <- apply(sc_mat, 2, z_col)

# Relabel Unknowns by max-z with a margin
lab <- as.character(tumor_ecm$comprehensive_cell_type)
lab[is.na(lab) | lab == ""] <- "Unknown"
unk_idx <- which(lab == "Unknown")

if (length(unk_idx)) {
  z_sub <- sc_z[unk_idx, , drop = FALSE]
  best_j   <- apply(z_sub, 1, which.max)
  best_val <- z_sub[cbind(seq_along(best_j), best_j)]
  # 2nd best (protect if only one finite value)
  second_val <- apply(z_sub, 1, function(x) {
    x <- sort(x[is.finite(x)], decreasing = TRUE)
    if (length(x) >= 2) x[2] else 0
  })
  margin <- best_val - second_val

  # Require reasonable separation
  keep <- which(is.finite(best_val) & best_val > 0.5 & margin > 0.2)
  relabeled <- unk_idx[keep]
  lab_names <- colnames(z_sub)[best_j[keep]]

  lab[relabeled] <- lab_names
  tumor_ecm$comprehensive_cell_type <- lab

  cat("Unknown salvaged in ECM-only:", length(relabeled), "\n")
  print(sort(table(lab), decreasing = TRUE))
} else {
  cat("No Unknown cells in ECM-only.\n")
}

# Save updated subset
ecm_dir <- "/content/drive/MyDrive/BayesPrism/two_phase_results_ecm"
dir.create(ecm_dir, recursive = TRUE, showWarnings = FALSE)
saveRDS(tumor_ecm, file.path(ecm_dir, "tumor_ecm_subset_salvaged.rds"))
write.csv(tumor_ecm@meta.data, file.path(ecm_dir, "seurat_metadata_ecm_salvaged.csv"))


In [ ]:
%%R
# STEP 5S — Quick UMAP for ECM-only
DefaultAssay(tumor_ecm) <- "RNA"
tumor_ecm <- FindVariableFeatures(tumor_ecm, selection.method = "vst", nfeatures = 2000, verbose = FALSE)
tumor_ecm <- ScaleData(tumor_ecm, features = VariableFeatures(tumor_ecm), verbose = FALSE)
tumor_ecm <- RunPCA(tumor_ecm, npcs = 40, verbose = FALSE)
tumor_ecm <- FindNeighbors(tumor_ecm, dims = 1:30, verbose = FALSE)
tumor_ecm <- FindClusters(tumor_ecm, resolution = 0.4, verbose = FALSE)
tumor_ecm <- RunUMAP(tumor_ecm, dims = 1:30, verbose = FALSE)

p1 <- DimPlot(tumor_ecm, group.by = "comprehensive_cell_type", label = TRUE, repel = TRUE) + NoLegend()
p2 <- DimPlot(tumor_ecm, group.by = "sample_id")
print(p1); print(p2)

saveRDS(tumor_ecm, file.path(ecm_dir, "tumor_ecm_subset_umap.rds"))


In [ ]:
%%R
# STEP 5T — Sanity: immune/heme contamination check in ECM-only
imm_g  <- negative_markers$immune_cd45
rbc_g  <- c("HBA1","HBA2","HBB","ALAS2")
plt_g  <- c("ITGA2B","PPBP","PF4","TUBB1")
mast_g <- negative_markers$mast

imm_s  <- score_sig(imm_g, expr_ecm)
rbc_s  <- score_sig(rbc_g, expr_ecm)
plt_s  <- score_sig(plt_g, expr_ecm)
mast_s <- score_sig(mast_g, expr_ecm)

qc_df <- data.frame(
  cell = colnames(tumor_ecm),
  ImmuneScore = imm_s,
  RBCScore    = rbc_s,
  Platelet    = plt_s,
  Mast        = mast_s,
  type        = tumor_ecm$comprehensive_cell_type,
  stringsAsFactors = FALSE
)

write.csv(qc_df, file.path(ecm_dir, "ecm_contamination_scores.csv"), row.names = FALSE)
apply(qc_df[,2:5] > quantile(qc_df$ImmuneScore, 0.98, na.rm = TRUE), 2, sum)


In [ ]:
%%R
# STEP 5N — Save final annotations
cat("=== STEP 5N: Save final annotations ===\n")
out_dir <- if (exists("ECM_RUN_DIR")) ECM_RUN_DIR else "/content/drive/MyDrive/BayesPrism/two_phase_results_ecm"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
write.csv(tumor_combined@meta.data,
          file.path(out_dir, "seurat_metadata_with_comprehensive_types_FINAL2.csv"))
cnt <- as.data.frame(sort(table(tumor_combined$comprehensive_cell_type), decreasing = TRUE))
write.csv(cnt, file.path(out_dir, "celltype_counts_FINAL2.csv"), row.names = FALSE)
print(cnt)


In [ ]:
%%R
suppressPackageStartupMessages({ library(Seurat); library(Matrix) })
cat("=== BUILD SINGLE REFERENCE FROM Seurat ===\n")

# Where to save
results_dir <- "/content/drive/MyDrive/BayesPrism/single_reference"
dir.create(results_dir, recursive = TRUE, showWarnings = FALSE)

stopifnot(exists("tumor_combined"))
stopifnot("comprehensive_cell_type" %in% colnames(tumor_combined@meta.data))

# Ensure layers are joined (Seurat v5 safety) and pull RAW counts
suppressWarnings(tumor_combined <- tryCatch(JoinLayers(tumor_combined, layers = NULL),
                                            error = function(e) tumor_combined))
ref_counts <- tryCatch(GetAssayData(tumor_combined, layer = "counts"),
                       error = function(e) GetAssayData(tumor_combined, slot = "counts"))

# Final labels from step 5N
labels <- as.character(tumor_combined$comprehensive_cell_type)
names(labels) <- colnames(tumor_combined)

drop_types <- c("Erythrocytes","Platelets","Low_quality")
keep <- !(labels %in% drop_types) & !is.na(labels)
ref_counts <- ref_counts[, keep, drop = FALSE]
ref_labels <- labels[keep]

# Collapse duplicated gene symbols by summing rows (both BP & bulk prefer one row per gene)
collapse_dups <- function(m) {
  gn <- rownames(m)
  if (anyDuplicated(gn)) {
    out <- rowsum(m, gn, reorder = FALSE)
    out[order(rownames(out)), , drop = FALSE]
  } else m
}
ref_counts <- collapse_dups(ref_counts)

saveRDS(ref_counts, file.path(results_dir, "reference_counts.rds"))
saveRDS(ref_labels, file.path(results_dir, "reference_labels.rds"))

cat("Saved reference:\n  genes =", nrow(ref_counts), " | cells =", ncol(ref_counts), "\n")
print(sort(table(ref_labels), decreasing = TRUE))


In [ ]:
%%R


# Load required libraries
library(BayesPrism)
library(dplyr)
library(ggplot2)
library(data.table)

# Required for statistical modeling
if (!requireNamespace("betareg", quietly = TRUE)) {
  install.packages("betareg")
}
if (!requireNamespace("compositions", quietly = TRUE)) {
  install.packages("compositions")
}

library(betareg)
library(compositions)


In [ ]:
%%R
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

BiocManager::install(c("org.Hs.eg.db", "AnnotationDbi"))

library(org.Hs.eg.db)
library(AnnotationDbi)

In [ ]:
%%R
# IMPROVED BAYESPRISM WORKFLOW WITH PROPER SAMPLE ID MATCHING
# Based on data exploration results - handles complex sample ID patterns

# Load required libraries
suppressPackageStartupMessages({
  library(BayesPrism)
  library(Matrix)
  library(org.Hs.eg.db)
  library(AnnotationDbi)
  library(ggplot2)
  library(dplyr)
})

cat("=== IMPROVED BAYESPRISM WORKFLOW ===\n")
cat("Based on data exploration findings:\n")
cat("- BMI sample IDs: 'C3L-00589' format with spaces\n")
cat("- Expression sample IDs: 'C3L-00589-01;C3L-00589-02' format\n")
cat("- Need custom pattern matching\n\n")

# Create results directory
results_dir <- "/content/drive/MyDrive/BayesPrism/improved_deconvolution_results"
dir.create(results_dir, recursive = TRUE, showWarnings = FALSE)

# STEP 1: LOAD AND CLEAN BMI DATA

cat("=== STEP 1: LOADING AND CLEANING BMI DATA ===\n")

# Load BMI data
bmi_file <- "/content/drive/MyDrive/BayesPrism/data/sample_id_with_BMI.csv"
bmi_data <- read.csv(bmi_file, stringsAsFactors = FALSE)

cat("BMI data loaded:\n")
cat("- Samples:", nrow(bmi_data), "\n")
cat("- Columns:", paste(colnames(bmi_data), collapse = ", "), "\n")

# Clean BMI sample IDs (remove leading/trailing spaces)
bmi_data$sample_id <- trimws(bmi_data$sample_id)
bmi_sample_ids <- bmi_data$sample_id

cat("BMI sample ID examples (cleaned):", paste(head(bmi_sample_ids, 3), collapse = ", "), "\n")
cat("BMI sample ID lengths:", paste(unique(nchar(bmi_sample_ids)), collapse = ", "), "\n")

# Create BMI groups from continuous BMI values
bmi_data$BMI_group <- cut(bmi_data$bmi,
                         breaks = c(0, 25, 30, Inf),
                         labels = c("Normal", "Overweight", "Obese"),
                         right = FALSE)

cat("BMI group distribution:\n")
print(table(bmi_data$BMI_group))

# STEP 2: LOAD EXPRESSION DATA WITH IMPROVED SAMPLE ID HANDLING

cat("\n=== STEP 2: LOADING EXPRESSION DATA ===\n")

# File paths
expr_files <- list(
  normal = "/content/drive/MyDrive/BayesPrism/data/pdac_normal_weight.csv",
  overweight = "/content/drive/MyDrive/BayesPrism/data/pdac_over_weight.csv",
  obese = "/content/drive/MyDrive/BayesPrism/data/pdac_Obese.csv"
)

# Function to load and clean expression data
load_and_clean_expression <- function(filepath, group_name) {
  cat("Loading", group_name, "data...\n")

  data <- read.csv(filepath, check.names = FALSE, stringsAsFactors = FALSE)

  # Extract gene IDs and expression matrix
  gene_ids <- data[, 1]
  expr_matrix <- as.matrix(data[, -1])
  rownames(expr_matrix) <- gene_ids
  mode(expr_matrix) <- "numeric"

  # Clean sample IDs in column names
  original_sample_ids <- colnames(expr_matrix)

  cat("- Original samples:", ncol(expr_matrix), "\n")
  cat("- Sample ID examples:", paste(head(original_sample_ids, 2), collapse = ", "), "\n")

  return(list(
    matrix = expr_matrix,
    original_ids = original_sample_ids,
    group = group_name
  ))
}

# Load all expression data
expr_data_list <- list()
for (group in names(expr_files)) {
  expr_data_list[[group]] <- load_and_clean_expression(expr_files[[group]], group)
}

# STEP 3: IMPROVED SAMPLE ID MATCHING

cat("\n=== STEP 3: IMPROVED SAMPLE ID MATCHING ===\n")

# Function to extract base sample ID from complex expression sample IDs
extract_base_sample_id_improved <- function(expr_sample_ids) {

  # Function to process a single sample ID
  process_single_id <- function(sample_id) {
    # Step 1: If semicolon-separated, take the first one
    if (grepl(";", sample_id)) {
      first_part <- strsplit(sample_id, ";")[[1]][1]
    } else {
      first_part <- sample_id
    }

    # Step 2: Remove suffix (-01, -02, -03, etc.)
    base_id <- gsub("-\\d+$", "", first_part)

    return(base_id)
  }

  # Apply to all sample IDs
  base_ids <- sapply(expr_sample_ids, process_single_id, USE.NAMES = FALSE)

  return(base_ids)
}

# Test the improved extraction on all expression sample IDs
all_expr_sample_ids <- c()
sample_group_mapping <- c()

for (group in names(expr_data_list)) {
  group_sample_ids <- expr_data_list[[group]]$original_ids
  all_expr_sample_ids <- c(all_expr_sample_ids, group_sample_ids)
  sample_group_mapping <- c(sample_group_mapping, rep(group, length(group_sample_ids)))
}

# Extract base IDs using improved method
base_ids_extracted <- extract_base_sample_id_improved(all_expr_sample_ids)

# Check matches with BMI data
matches <- sum(base_ids_extracted %in% bmi_sample_ids)
match_rate <- matches / length(base_ids_extracted)

cat("Improved sample ID matching results:\n")
cat("- Total expression samples:", length(all_expr_sample_ids), "\n")
cat("- Matches with BMI data:", matches, "\n")
cat("- Match rate:", round(match_rate * 100, 1), "%\n")

# Show examples of the transformation
cat("\nExample transformations:\n")
for (i in 1:min(5, length(all_expr_sample_ids))) {
  cat("'", all_expr_sample_ids[i], "' → '", base_ids_extracted[i], "'\n", sep = "")
}

matched_bmi_ids <- base_ids_extracted[base_ids_extracted %in% bmi_sample_ids]
cat("\nMatched BMI sample IDs (first 5):\n")
print(head(sort(unique(matched_bmi_ids)), 5))

if (match_rate < 0.7) {
  warning("Match rate is below 70%. Manual inspection recommended.")
} else {
  cat(" Good match rate achieved!\n")
}

# STEP 4: CREATE SAMPLE MAPPING TABLE

cat("\n=== STEP 4: CREATING COMPREHENSIVE SAMPLE MAPPING ===\n")

# Create comprehensive sample mapping
sample_mapping <- data.frame(
  expr_sample_id = all_expr_sample_ids,
  base_sample_id = base_ids_extracted,
  expression_group = sample_group_mapping,
  in_bmi_data = base_ids_extracted %in% bmi_sample_ids,
  stringsAsFactors = FALSE
)

# Add BMI information for matched samples
sample_mapping$BMI <- NA
sample_mapping$BMI_group <- NA
sample_mapping$BMI_group_from_expr <- sample_mapping$expression_group  # Keep original grouping

for (i in 1:nrow(sample_mapping)) {
  base_id <- sample_mapping$base_sample_id[i]
  if (base_id %in% bmi_sample_ids) {
    bmi_row <- which(bmi_sample_ids == base_id)[1]
    sample_mapping$BMI[i] <- bmi_data$bmi[bmi_row]
    sample_mapping$BMI_group[i] <- as.character(bmi_data$BMI_group[bmi_row])
  }
}

# Check consistency between expression groups and BMI groups
cat("Sample mapping summary:\n")
cat("- Total samples:", nrow(sample_mapping), "\n")
cat("- Samples with BMI data:", sum(sample_mapping$in_bmi_data), "\n")
cat("- Samples with BMI values:", sum(!is.na(sample_mapping$BMI)), "\n")

# Compare expression grouping vs BMI grouping
if (sum(!is.na(sample_mapping$BMI_group)) > 0) {
  cat("\nComparison: Expression groups vs BMI-derived groups:\n")
  comparison_table <- table(sample_mapping$expression_group,
                           sample_mapping$BMI_group,
                           useNA = "ifany")
  print(comparison_table)

  # Calculate agreement
  matched_samples <- sample_mapping[!is.na(sample_mapping$BMI_group), ]
  if (nrow(matched_samples) > 0) {
    agreement <- sum(matched_samples$expression_group == matched_samples$BMI_group) / nrow(matched_samples)
    cat("Agreement between expression and BMI grouping:", round(agreement * 100, 1), "%\n")
  }
}

# Save sample mapping
write.csv(sample_mapping, file.path(results_dir, "detailed_sample_mapping.csv"), row.names = FALSE)

# STEP 5: COMBINE EXPRESSION MATRICES

cat("\n=== STEP 5: COMBINING EXPRESSION MATRICES ===\n")

# Get all gene IDs to find common genes
all_gene_lists <- lapply(expr_data_list, function(x) rownames(x$matrix))
common_genes <- Reduce(intersect, all_gene_lists)

cat("Gene intersection:\n")
for (group in names(all_gene_lists)) {
  cat("- ", group, ": ", length(all_gene_lists[[group]]), " genes\n", sep = "")
}
cat("- Common genes:", length(common_genes), "\n")

# Combine matrices using common genes
combined_matrices <- list()
for (group in names(expr_data_list)) {
  matrix_filtered <- expr_data_list[[group]]$matrix[common_genes, ]
  combined_matrices[[group]] <- matrix_filtered
}

# Merge all matrices
combined_expr <- do.call(cbind, combined_matrices)

cat("Combined expression matrix:\n")
cat("- Genes:", nrow(combined_expr), "\n")
cat("- Samples:", ncol(combined_expr), "\n")

stopifnot(all(colnames(combined_expr) == sample_mapping$expr_sample_id))

# STEP 6: GENE ID PROCESSING

cat("\n=== STEP 6: PROCESSING GENE IDs ===\n")

# Remove version numbers from Ensembl IDs
original_gene_ids <- rownames(combined_expr)
clean_ensembl_ids <- gsub("\\.\\d+$", "", original_gene_ids)

cat("Gene ID processing:\n")
cat("- Original genes:", length(original_gene_ids), "\n")
cat("- All have version numbers:", all(grepl("\\.", original_gene_ids)), "\n")
cat("- Duplicates after cleaning:", sum(duplicated(clean_ensembl_ids)), "\n")

# Update matrix with clean IDs
rownames(combined_expr) <- clean_ensembl_ids

# Map to gene symbols
cat("Mapping Ensembl IDs to gene symbols...\n")
ensembl_to_symbol <- mapIds(org.Hs.eg.db,
                           keys = clean_ensembl_ids,
                           column = "SYMBOL",
                           keytype = "ENSEMBL",
                           multiVals = "first")

# Filter to genes with symbols
genes_with_symbols <- !is.na(ensembl_to_symbol) & ensembl_to_symbol != ""
combined_expr_symbols <- combined_expr[genes_with_symbols, ]
rownames(combined_expr_symbols) <- ensembl_to_symbol[genes_with_symbols]

cat("Gene symbol mapping:\n")
cat("- Genes with symbols:", sum(genes_with_symbols), "\n")
cat("- Mapping rate:", round(sum(genes_with_symbols)/length(genes_with_symbols)*100, 1), "%\n")

# Handle duplicated gene symbols
if (sum(duplicated(rownames(combined_expr_symbols))) > 0) {
  cat("Handling", sum(duplicated(rownames(combined_expr_symbols))), "duplicated gene symbols...\n")

  # For each duplicated symbol, keep the gene with highest mean expression
  gene_symbols <- rownames(combined_expr_symbols)
  unique_symbols <- unique(gene_symbols)

  final_expr <- matrix(0, nrow = length(unique_symbols), ncol = ncol(combined_expr_symbols))
  rownames(final_expr) <- unique_symbols
  colnames(final_expr) <- colnames(combined_expr_symbols)

  for (symbol in unique_symbols) {
    symbol_rows <- which(gene_symbols == symbol)
    if (length(symbol_rows) == 1) {
      final_expr[symbol, ] <- combined_expr_symbols[symbol_rows, ]
    } else {
      # Keep row with highest mean expression
      row_means <- rowMeans(combined_expr_symbols[symbol_rows, , drop = FALSE])
      best_row <- symbol_rows[which.max(row_means)]
      final_expr[symbol, ] <- combined_expr_symbols[best_row, ]
    }
  }

  combined_expr_final <- final_expr
} else {
  combined_expr_final <- combined_expr_symbols
}

cat("Final expression matrix:", nrow(combined_expr_final), "genes x", ncol(combined_expr_final), "samples\n")

# STEP 7: LOAD REFERENCE AND MATCH GENES

cat("\n=== STEP 7: LOADING REFERENCE AND MATCHING GENES ===\n")

# Load reference
ref_counts <- readRDS("/content/drive/MyDrive/BayesPrism/single_reference/reference_counts.rds")
ref_labels <- readRDS("/content/drive/MyDrive/BayesPrism/single_reference/reference_labels.rds")

cat("Reference loaded:\n")
cat("- Genes:", nrow(ref_counts), "\n")
cat("- Cells:", ncol(ref_counts), "\n")
cat("- Cell types:", length(unique(ref_labels)), "\n")

# Find common genes
common_genes_ref_bulk <- intersect(rownames(ref_counts), rownames(combined_expr_final))
cat("Common genes between reference and bulk:", length(common_genes_ref_bulk), "\n")

if (length(common_genes_ref_bulk) < 5000) {
  warning("Low gene overlap (", length(common_genes_ref_bulk), "). Results may be suboptimal.")
}

# Filter to common genes
ref_filtered <- ref_counts[common_genes_ref_bulk, ]
bulk_filtered <- combined_expr_final[common_genes_ref_bulk, ]

# Convert to matrices and ensure non-negative
ref_filtered <- as.matrix(ref_filtered)
bulk_filtered <- as.matrix(bulk_filtered)
ref_filtered[ref_filtered < 0] <- 0
bulk_filtered[bulk_filtered < 0] <- 0

cat("Final aligned matrices:\n")
cat("- Reference:", nrow(ref_filtered), "x", ncol(ref_filtered), "\n")
cat("- Bulk:", nrow(bulk_filtered), "x", ncol(bulk_filtered), "\n")

# STEP 8: RUN BAYESPRISM

cat("\n=== STEP 8: RUNNING BAYESPRISM DECONVOLUTION ===\n")

# Transpose for BayesPrism (needs cells/samples x genes)
sc_ref_t <- t(ref_filtered)
bulk_mix_t <- t(bulk_filtered)

# Prepare cell type labels
cell_type_labels <- as.character(ref_labels)

cat("BayesPrism setup:\n")
cat("- Reference (cells x genes):", dim(sc_ref_t), "\n")
cat("- Bulk (samples x genes):", dim(bulk_mix_t), "\n")
cat("- Cell types:", length(unique(cell_type_labels)), "\n")

# Create BayesPrism object
set.seed(42)
bp <- new.prism(
  reference = sc_ref_t,
  mixture = bulk_mix_t,
  input.type = "count",
  cell.type.labels = cell_type_labels,
  cell.state.labels = cell_type_labels,
  key = NULL,
  outlier.cut = 0.01,
  outlier.fraction = 0.05
)

cat("Running BayesPrism deconvolution...\n")
bp <- run.prism(
  bp,
  n.cores = 6,
  update.gibbs = TRUE,
  gibbs.control = list(
    chain.length = 2000,
    burn.in = 500,
    thinning = 2
  )
)

cat("BayesPrism deconvolution completed!\n")

# STEP 9: EXTRACT AND SAVE RESULTS

cat("\n=== STEP 9: EXTRACTING AND SAVING RESULTS ===\n")

# Extract cell type fractions
theta_final <- get.fraction(bp, which.theta = "final", state.or.type = "type")

# Ensure correct orientation (samples x cell_types)
if (nrow(theta_final) != ncol(bulk_filtered)) {
  theta_final <- t(theta_final)
}

# Create comprehensive results
results_final <- cbind(sample_mapping, theta_final)

cat("Results summary:\n")
cat("- Samples:", nrow(results_final), "\n")
cat("- Cell types:", ncol(theta_final), "\n")
cat("- Samples with BMI data:", sum(results_final$in_bmi_data), "\n")

# Save all results
saveRDS(bp, file.path(results_dir, "bayesprism_object.rds"))
saveRDS(theta_final, file.path(results_dir, "cell_type_proportions.rds"))
saveRDS(results_final, file.path(results_dir, "complete_results.rds"))
write.csv(results_final, file.path(results_dir, "complete_results.csv"), row.names = FALSE)

# Save processing info
processing_info <- list(
  sample_id_match_rate = match_rate,
  total_samples = ncol(combined_expr_final),
  samples_with_bmi = sum(sample_mapping$in_bmi_data),
  total_genes_final = nrow(combined_expr_final),
  common_genes_with_ref = length(common_genes_ref_bulk),
  cell_types = length(unique(ref_labels)),
  bmi_expr_agreement = if(exists("agreement")) agreement else NA
)

saveRDS(processing_info, file.path(results_dir, "processing_info.rds"))

cat("\n🎉 COMPLETE WORKFLOW FINISHED SUCCESSFULLY! 🎉\n")
cat("Results saved to:", results_dir, "\n")
cat("\nKey achievements:\n")
cat("- Sample ID matching:", round(match_rate*100, 1), "% success rate\n")
cat("- Gene mapping:", nrow(combined_expr_final), "genes with symbols\n")
cat("- Reference overlap:", length(common_genes_ref_bulk), "common genes\n")
cat("- Deconvolution:", ncol(theta_final), "cell types analyzed\n")

cat("\nFiles created:\n")
cat("- complete_results.csv: Full results with BMI and cell proportions\n")
cat("- bayesprism_object.rds: Complete BayesPrism object\n")
cat("- detailed_sample_mapping.csv: Sample ID mapping details\n")
cat("- processing_info.rds: Summary statistics\n")

In [ ]:
%%R
# COMPREHENSIVE BayesPrism Expression Extraction
# Extracts both TYPE and STATE levels from all 3 BayesPrism objects
# Saves everything to Google Drive in organized structure

# Load required libraries
library(BayesPrism)
library(dplyr)

# Set working directory and create master output folder
setwd("/content")
master_dir <- "/content/drive/MyDrive/BayesPrism/3decon"
dir.create(master_dir, recursive = TRUE, showWarnings = FALSE)

cat("COMPREHENSIVE BayesPrism EXPRESSION EXTRACTION\n")
cat(strrep("=", 80), "\n")
cat("Master output directory:", master_dir, "\n")

# Define the three BayesPrism files and their configurations
bp_files <- list(
  list(
    file = "/content/bayesprism_immune_course.rds",
    name = "immune_course",
    description = "Coarse immune cell classification"
  ),
  list(
    file = "/content/bayesprism_immune_fine.rds",
    name = "immune_fine",
    description = "Fine-grained immune cell states (23 refined subtypes)"
  ),
  list(
    file = "/content/bayesprism_non_immune.rds",
    name = "non_immune",
    description = "Non-immune cell types"
  )
)

# Comprehensive extraction function
extract_comprehensive_csv <- function(bp_info) {

  cat("\n", strrep("=", 80), "\n")
  cat("🔬 PROCESSING:", bp_info$name, "\n")
  cat("📄 Description:", bp_info$description, "\n")
  cat("📁 File:", bp_info$file, "\n")
  cat(strrep("=", 80), "\n")

  # Check if file exists
  if (!file.exists(bp_info$file)) {
    cat("File not found:", bp_info$file, "\n")
    return(NULL)
  }

  # Create analysis-specific directory
  analysis_dir <- file.path(master_dir, bp_info$name)
  dir.create(analysis_dir, recursive = TRUE, showWarnings = FALSE)

  # Load BayesPrism object
  cat("📥 Loading BayesPrism object...\n")
  bp <- readRDS(bp_info$file)
  cat("✅ Object loaded successfully (Class:", class(bp), ")\n")

  # Initialize results tracking
  extraction_results <- list(
    analysis_name = bp_info$name,
    description = bp_info$description,
    source_file = bp_info$file,
    output_directory = analysis_dir,
    timestamp = Sys.time(),
    extractions = list()
  )

  # Function to extract and save expression data with smart detection
  extract_level <- function(level_name, state_or_type) {

    cat("\n--- CHECKING", toupper(level_name), "LEVEL ---\n")

    tryCatch({
      # Extract Z matrix
      Z_matrix <- get.exp(bp, state.or.type = state_or_type)
      cat( level_name, "matrix extracted successfully\n")
      cat(" Dimensions:", dim(Z_matrix), "[samples, genes,", level_name, "]\n")

      # Get names
      entity_names <- dimnames(Z_matrix)[[3]]
      sample_names <- dimnames(Z_matrix)[[1]]
      gene_names <- dimnames(Z_matrix)[[2]]

      n_entities <- length(entity_names)
      n_samples <- length(sample_names)
      n_genes <- length(gene_names)

      # Check if this level is different from others (avoid duplicates)
      skip_extraction <- FALSE
      if (level_name == "state" && exists("type_entities")) {
        if (identical(sort(entity_names), sort(type_entities))) {
          cat("⚠️  STATE level identical to TYPE level - skipping duplicate extraction\n")
          skip_extraction <- TRUE
        }
      }

      if (!skip_extraction) {
        cat("Found", n_entities, level_name, "entities:\n")
        for(i in 1:min(10, length(entity_names))) {  # Show first 10
          cat(sprintf("   %2d. %s\n", i, entity_names[i]))
        }
        if (n_entities > 10) {
          cat(sprintf("   ... and %d more\n", n_entities - 10))
        }

        cat("Matrix details:\n")
        cat("   Samples:", n_samples, "\n")
        cat("   Genes:", n_genes, "\n")
        cat("   Entities:", n_entities, "\n")

        # Store entity names for comparison
        if (level_name == "type") {
          type_entities <<- entity_names
        }

        # Create level-specific directory
        level_dir <- file.path(analysis_dir, paste0(level_name, "_level"))
        dir.create(level_dir, recursive = TRUE, showWarnings = FALSE)

        # Extract and save expression for each entity
        cat("Extracting expression for each", level_name, "...\n")

        success_count <- 0
        entity_file_mapping <- data.frame(
          entity_number = integer(),
          original_name = character(),
          clean_filename = character(),
          csv_path = character(),
          stringsAsFactors = FALSE
        )

        for (i in 1:n_entities) {
          entity_name <- entity_names[i]

          tryCatch({
            # Extract expression for this entity (samples x genes)
            entity_exp_3d <- Z_matrix[, , i, drop = FALSE]

            # Convert to 2D matrix (samples x genes)
            entity_exp_matrix <- matrix(entity_exp_3d,
                                       nrow = dim(entity_exp_3d)[1],
                                       ncol = dim(entity_exp_3d)[2])

            # Set row and column names
            rownames(entity_exp_matrix) <- sample_names
            colnames(entity_exp_matrix) <- gene_names

            # Create clean filename
            clean_entity_name <- gsub("[^A-Za-z0-9_]", "_", entity_name)
            clean_entity_name <- gsub("__+", "_", clean_entity_name)
            clean_entity_name <- gsub("^_|_$", "", clean_entity_name)

            # Create dataframe with Sample column for CSV export
            entity_df <- data.frame(
              Sample = rownames(entity_exp_matrix),
              entity_exp_matrix,
              stringsAsFactors = FALSE,
              check.names = FALSE
            )

            # Save as CSV
            csv_filename <- paste0(clean_entity_name, "_expression.csv")
            csv_path <- file.path(level_dir, csv_filename)

            write.csv(entity_df, csv_path, row.names = FALSE)

            # Track successful extraction
            entity_file_mapping <- rbind(entity_file_mapping, data.frame(
              entity_number = i,
              original_name = entity_name,
              clean_filename = csv_filename,
              csv_path = csv_path,
              stringsAsFactors = FALSE
            ))

            success_count <- success_count + 1
            if (i <= 5 || i %% 5 == 0 || i == n_entities) {  # Show progress
              cat(sprintf("   %2d/%2d:  %s -> %s\n", i, n_entities, entity_name, csv_filename))
            }

          }, error = function(e) {
            cat(sprintf("   %2d/%2d:  %s -> Error: %s\n", i, n_entities, entity_name, e$message))
          })
        }

        # Save mapping file
        write.csv(entity_file_mapping, file.path(level_dir, paste0(level_name, "_filename_mapping.csv")), row.names = FALSE)

        # Create level summary
        level_summary <- list(
          level_name = level_name,
          level_directory = level_dir,
          total_entities = n_entities,
          successful_extractions = success_count,
          n_samples = n_samples,
          n_genes = n_genes,
          entity_names = entity_names,
          entity_file_mapping = entity_file_mapping
        )

        # Save level summary
        saveRDS(level_summary, file.path(level_dir, paste0(level_name, "_summary.rds")))

        # Create readable summary
        level_summary_text <- paste0(
          "BayesPrism ", toupper(level_name), " Level Extraction\n",
          strrep("=", 40), "\n",
          "Analysis: ", bp_info$name, "\n",
          "Level: ", toupper(level_name), "\n",
          "Total entities: ", n_entities, "\n",
          "Successfully extracted: ", success_count, "\n",
          "Samples: ", n_samples, "\n",
          "Genes: ", n_genes, "\n",
          "Timestamp: ", Sys.time(), "\n\n",
          "Entities extracted:\n",
          paste(paste0("  ", 1:length(entity_names), ". ", entity_names), collapse = "\n"), "\n\n",
          "Files created:\n",
          paste(paste0("  - ", entity_file_mapping$clean_filename), collapse = "\n"), "\n"
        )

        writeLines(level_summary_text, file.path(level_dir, paste0(level_name, "_summary.txt")))

        cat("🎉 Completed", level_name, "level - Successfully saved", success_count, "out of", n_entities, "entities\n")

        return(level_summary)
      } else {
        cat("Skipping duplicate", level_name, "level extraction\n")
        return(NULL)
      }

    }, error = function(e) {
      cat( level_name, "level not available:", e$message, "\n")
      return(NULL)
    })
  }

  # Reset comparison variable
  type_entities <- NULL

  # Always try TYPE level first
  type_result <- extract_level("type", "type")
  if (!is.null(type_result)) {
    extraction_results$extractions$type <- type_result
  }

  # Try STATE level (especially important for immune_fine)
  state_result <- extract_level("state", "state")
  if (!is.null(state_result)) {
    extraction_results$extractions$state <- state_result
  }

  # Save overall analysis summary
  saveRDS(extraction_results, file.path(analysis_dir, "comprehensive_extraction_summary.rds"))

  # Create overall analysis summary text
  overall_text <- paste0(
    "BayesPrism Comprehensive Extraction Summary\n",
    strrep("=", 48), "\n",
    "Analysis: ", bp_info$name, "\n",
    "Description: ", bp_info$description, "\n",
    "Source file: ", bp_info$file, "\n",
    "Output directory: ", analysis_dir, "\n",
    "Timestamp: ", Sys.time(), "\n\n",
    "Extraction Results:\n"
  )

  if (!is.null(extraction_results$extractions$type)) {
    type_info <- extraction_results$extractions$type
    overall_text <- paste0(overall_text,
      " TYPE Level: ", type_info$total_entities, " entities, ",
      type_info$successful_extractions, " CSV files created\n"
    )
  } else {
    overall_text <- paste0(overall_text, " TYPE Level: Failed\n")
  }

  if (!is.null(extraction_results$extractions$state)) {
    state_info <- extraction_results$extractions$state
    overall_text <- paste0(overall_text,
      "STATE Level: ", state_info$total_entities, " entities, ",
      state_info$successful_extractions, " CSV files created\n"
    )
  } else {
    overall_text <- paste0(overall_text, " STATE Level: Failed or not available\n")
  }

  overall_text <- paste0(overall_text, "\nDirectory Structure:\n")
  overall_text <- paste0(overall_text, bp_info$name, "/\n")
  if (!is.null(extraction_results$extractions$type)) {
    overall_text <- paste0(overall_text, "├── type_level/\n")
  }
  if (!is.null(extraction_results$extractions$state)) {
    overall_text <- paste0(overall_text, "├── state_level/\n")
  }
  overall_text <- paste0(overall_text, "└── comprehensive_extraction_summary.rds\n")

  writeLines(overall_text, file.path(analysis_dir, "comprehensive_extraction_summary.txt"))

  return(extraction_results)
}

# Process all three BayesPrism files
cat("Starting comprehensive extraction for all 3 datasets...\n")

all_results <- list()
total_csv_files <- 0

for (bp_info in bp_files) {
  result <- extract_comprehensive_csv(bp_info)
  if (!is.null(result)) {
    all_results[[bp_info$name]] <- result

    # Count CSV files
    if (!is.null(result$extractions$type)) {
      total_csv_files <- total_csv_files + result$extractions$type$successful_extractions
    }
    if (!is.null(result$extractions$state)) {
      total_csv_files <- total_csv_files + result$extractions$state$successful_extractions
    }
  }
}

# Create master summary
cat("\n", strrep("=", 80), "\n")
cat("🎯 COMPREHENSIVE EXTRACTION COMPLETED!\n")
cat(strrep("=", 80), "\n")

for (name in names(all_results)) {
  result <- all_results[[name]]
  cat(sprintf("📊 %-15s: ", name))

  type_count <- if (!is.null(result$extractions$type)) result$extractions$type$successful_extractions else 0
  state_count <- if (!is.null(result$extractions$state)) result$extractions$state$successful_extractions else 0

  cat(sprintf("TYPE: %2d, STATE: %2d CSV files\n", type_count, state_count))
}

cat("\n📈 Total CSV files created:", total_csv_files, "\n")

# Create master directory structure description
cat("\n📁 Master directory structure:\n")
cat("3decon/\n")
for (name in names(all_results)) {
  result <- all_results[[name]]
  cat("├──", name, "/\n")
  if (!is.null(result$extractions$type)) {
    cat("│   ├── type_level/ (", result$extractions$type$successful_extractions, " CSV files)\n")
  }
  if (!is.null(result$extractions$state)) {
    cat("│   ├── state_level/ (", result$extractions$state$successful_extractions, " CSV files)\n")
  }
  cat("│   └── summaries\n")
}

# Save master summary
master_summary <- list(
  timestamp = Sys.time(),
  master_directory = master_dir,
  total_csv_files = total_csv_files,
  analyses = all_results,
  bp_files_processed = bp_files
)

saveRDS(master_summary, file.path(master_dir, "master_extraction_summary.rds"))

# Create master text summary
master_text <- paste0(
  "BayesPrism Master Extraction Summary\n",
  strrep("=", 40), "\n",
  "Timestamp: ", Sys.time(), "\n",
  "Master directory: ", master_dir, "\n",
  "Total CSV files created: ", total_csv_files, "\n\n",
  "Analyses completed:\n"
)

for (name in names(all_results)) {
  result <- all_results[[name]]
  type_count <- if (!is.null(result$extractions$type)) result$extractions$type$successful_extractions else 0
  state_count <- if (!is.null(result$extractions$state)) result$extractions$state$successful_extractions else 0

  master_text <- paste0(master_text,
    sprintf("- %s: TYPE=%d, STATE=%d CSV files\n", name, type_count, state_count)
  )
}

master_text <- paste0(master_text, "\nNote: immune_fine has both TYPE (11 broad) and STATE (23 refined) levels.\n")
master_text <- paste0(master_text, "Other analyses may only have TYPE level available.\n")

writeLines(master_text, file.path(master_dir, "master_extraction_summary.txt"))

cat("\n📁 All files saved to Google Drive:", master_dir, "\n")
cat("📊 Each CSV contains: Sample column + gene expression values\n")
cat("🔬 Ready for downstream analysis!\n")

# Special note about immune_fine
if ("immune_fine" %in% names(all_results)) {
  if (!is.null(all_results$immune_fine$extractions$state)) {
    cat("\n🎯 SPECIAL: immune_fine has 23 refined cell states including:\n")
    cat("   - Monocytes classical, CD4+ T reg, CD8+ T exhausted\n")
    cat("   - TAM, mDC, CD8+ T effector, and more!\n")
  }
}

cat("\n🚀 COMPREHENSIVE EXTRACTION COMPLETE! 🚀\n")